# Once Upon a Matchmaker, in Python

In the game, the **Matchmaker** matches people with films the way a dating site matches people with people.
Every customer fills in a profile on three axes, real or fiction, comedy or drama, action or calm, and a film
critic describes every film on the same three axes. Turned into numbers, the profiles and the critic's answers
are the **embeddings**: three numbers for each customer and three for each film. The score of a customer for a
film is the inner product of their two rows, and all the scores together make the table of scores, the
**rating matrix**.

The notebook has four parts. Do each part when the game sends you here, after its section:

1. **Part 1, multiplying two matrices**, after section 1, "What the Matchmaker knows": one score, then one row,
   then the whole table of scores (about 30 minutes).
2. **Part 2, guess and correct**, after section 2, "Only the ratings": guess two rows, measure the loss, and let
   the computer play what-if until the rating matches.
3. **Part 3, gradient descent** (optional), after section 3, "Smaller hops": feel the slope with a tiny hop,
   then hop downhill.
4. **Part 4, every customer and every film**, after section 4, "Many customers, many films": find all the
   profiles from the ratings alone.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A
folded cell with a title holds plumbing: run it, and open it only if you are curious. The 💡 hints are folded
cells too: double-click one to open it, and open the next one only if you are still stuck. After each task, a
check cell tells you whether your code works.

# Part 1. Multiplying two matrices

In section 1 of the game the Matchmaker filled its whole table of scores from the profiles, one inner product for
each customer and each film, and the operation got its name: **matrix multiplication**. Now you teach it to a
computer, in small steps:

- **Task A**: one score, Leo and *Men in Black*;
- **Task B**: one row, a customer's scores for every film;
- **Task C**: the whole table, which is matrix multiplication;
- **Task D**: the rule that says when two matrices can be multiplied.

At the end your code multiplies the bigger pair of scene 4. Run the setup cell first.

In [ ]:
#@title Setup of part 1 (run this first) { display-mode: "form" }
# The plumbing of part 1: show_matrix, which draws a matrix as a table, and the checks of tasks A to D.
# You do not need to read it. A check prints one short message and never stops the notebook.
try:
    from IPython import get_ipython as _p1_get_ipython
    from IPython.display import display as _p1_display
except ImportError:
    _p1_get_ipython = None
    _p1_display = None

# The game's numbers, kept here as well, so that the checks still work if you change the lists in the notebook
_P1_CUSTOMER_MATRIX = [[2, 1, -2], [-2, 2, 2], [1, -2, 1]]
_P1_FILM_MATRIX = [[2, 1, -1], [2, -2, 1], [-2, 1, 2], [-2, -2, 0]]
_P1_CUSTOMER_NAMES = ['Sara', 'Leo', 'Maya']
_P1_FILM_NAMES = ['The Intouchables', 'Apollo 13', 'Men in Black', 'Interstellar']
_P1_AXES = ['real', 'comedy', 'action']
_P1_BIG_A = [[1, -2, 0, 2], [2, 1, -1, 0], [-1, 0, 2, 1], [0, 2, 1, -2], [2, -1, 1, 1]]
_P1_BIG_B = [[1, 1, 0, -1], [0, -2, 2, 1], [2, 0, -1, 1], [-1, 1, 1, 0], [1, 2, -2, 2], [0, -1, 1, -2]]
_P1_BIG_PRODUCT = [[-3, 6, 4, -3, 1, -2], [3, -4, 5, -2, 6, -2], [-2, 5, -3, 3, -3, 0], [4, -4, -3, 3, -2, 3], [0, 5, 4, -2, 0, 0]]
_P1_ONE = (1, 2)   # task A: Leo's row in customer_matrix, and the row of Men in Black in film_matrix


def _p1_inner(row_a, row_b):
    total = 0
    for i in range(len(row_a)):
        total = total + row_a[i] * row_b[i]
    return total


def _p1_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = repr(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 80:
        text = text[:77] + "..."
    return text


def _p1_and(items):
    """a, b and c"""
    items = [str(x) for x in items]
    if len(items) == 1:
        return items[0]
    return ", ".join(items[:-1]) + " and " + items[-1]


def _p1_count(n, word):
    """1 row, 3 rows"""
    if n == 1:
        return "1 " + word
    return "%d %ss" % (n, word)


def _p1_is_number(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)


# show_matrix: a matrix as a small table

def _p1_entry(x):
    """One entry of a table as text: 7, -2, 0.75 (two decimals for a decimal number), ? for None."""
    if x is None:
        return "?"
    if x is Ellipsis:
        return "..."
    if isinstance(x, bool):
        return str(x)
    if isinstance(x, float):
        return "%.2f" % x
    return str(x)


def _p1_tint(x):
    """The background of an entry, as in the game: blue above 0, orange below 0, grey for 0, and a stronger
    tint from 2 and -2 on. The tints are see-through, so the table reads in a light and in a dark theme."""
    if not _p1_is_number(x):
        return ""
    if x > 0:
        colour = "59,130,214"
    elif x < 0:
        colour = "224,116,42"
    else:
        return "background:rgba(140,140,140,0.16);"
    strength = 0.16
    if abs(x) >= 2:
        strength = 0.32
    return "background:rgba(%s,%.2f);" % (colour, strength)


def _p1_not_a_table(matrix):
    """None when matrix is a table (a list of rows of the same length, each entry a single value), else a pair
    (kind, reason): kind "wait" when a task is not finished yet, "shape" when the value has another shape."""
    if matrix is None:
        return ("wait", "there is no table yet, only None (does the function end with a return line?)")
    if matrix is Ellipsis:
        return ("wait", "the table is still ..., a gap in a 🎯 cell")
    if not isinstance(matrix, (list, tuple)):
        return ("shape", "this is a single value, %s, and a table is a list of rows" % _p1_short(matrix))
    if len(matrix) == 0:
        return ("wait", "the table is an empty list, []")
    for r in range(len(matrix)):
        row = matrix[r]
        if row is Ellipsis:
            return ("wait", "row %d of the table is still ..., a gap in a 🎯 cell" % r)
        if row is None:
            return ("wait", "row %d of the table is None (does the function that makes a row end with a return "
                            "line?)" % r)
        if not isinstance(row, (list, tuple)):
            return ("shape", "row %d is %s, a single value, and each row of a table is a list of numbers"
                    % (r, _p1_short(row)))
    lengths = []
    for row in matrix:
        lengths.append(len(row))
    if max(lengths) == 0:
        if len(matrix) == 1:
            return ("wait", "its only row is an empty list, []")
        return ("wait", "its rows are empty lists, []")
    if min(lengths) != max(lengths):
        return ("shape", "its rows have different lengths, %s, and the rows of a table are all equally long"
                % _p1_and(lengths))
    for row in matrix:
        for x in row:
            if x is Ellipsis:
                return ("wait", "it still holds a ..., a gap in a 🎯 cell")
            if isinstance(x, (list, tuple)):
                return ("shape", "its entries are lists, and each entry of a table is a single number")
    return None


def _p1_labels(names, n):
    """The labels of n rows or columns: the names when there is one name for each, the numbers from 0 when
    names is None, and None when the names do not fit."""
    if names is None:
        return [str(i) for i in range(n)]
    if isinstance(names, (list, tuple)) and len(names) == n:
        return [str(x) for x in names]
    return None


def show_matrix(matrix, row_names=None, column_names=None):
    """Show a matrix (a list of rows) as a table, with a name for each row and each column. Without names,
    the rows and the columns are numbered from 0, the way Python counts. When matrix is not a table yet
    (None, a gap, an empty list, rows of different lengths), it prints one calm line and stops nothing."""
    try:
        problem = _p1_not_a_table(matrix)
        if problem is not None:
            kind, reason = problem
            if kind == "wait":
                print("⏳ Nothing to show yet: %s. Finish the 🎯 cell above, run it, then run this cell again."
                      % reason)
            else:
                print("⏳ This is not a table yet: %s. The check of your function says more." % reason)
            return
        rows = len(matrix)
        columns = len(matrix[0])
        size = "%s and %s" % (_p1_count(rows, "row"), _p1_count(columns, "column"))
        row_labels = _p1_labels(row_names, rows)
        column_labels = _p1_labels(column_names, columns)
        if row_labels is None or column_labels is None:
            swapped = (isinstance(row_names, (list, tuple)) and isinstance(column_names, (list, tuple))
                       and len(row_names) == columns and len(column_names) == rows and rows != columns)
            if swapped:
                print("Note: this table has %s, and the names are for %s and %s. Are its rows and columns "
                      "swapped? They are numbered below." % (size, _p1_count(columns, "row"),
                                                              _p1_count(rows, "column")))
            else:
                print("Note: the names do not fit this table of %s, so it is numbered below." % size)
            row_labels = [str(i) for i in range(rows)]
            column_labels = [str(j) for j in range(columns)]

        entries = []
        for r in range(rows):
            line = []
            for c in range(columns):
                line.append(_p1_entry(matrix[r][c]))
            entries.append(line)

        # Plain text, for screens that cannot show the table
        name_width = max(len(label) for label in row_labels)
        widths = []
        for c in range(columns):
            width = len(column_labels[c])
            for r in range(rows):
                width = max(width, len(entries[r][c]))
            widths.append(width)
        lines = [" " * name_width + "".join("   " + column_labels[c].rjust(widths[c]) for c in range(columns))]
        for r in range(rows):
            lines.append(row_labels[r].ljust(name_width)
                         + "".join("   " + entries[r][c].rjust(widths[c]) for c in range(columns)))
        lines.append(size)
        text = "\n".join(line.rstrip() for line in lines)

        # The same table in HTML, each number tinted by its sign
        from html import escape
        edge = "border:1px solid rgba(128,128,128,0.3);"
        html = '<table style="border-collapse:collapse;font-size:15px;margin:6px 0 2px 0">'
        html += '<tr><th style="padding:4px 8px"></th>'
        for c in range(columns):
            html += '<th style="padding:4px 10px;text-align:center;font-weight:600">%s</th>' % escape(column_labels[c])
        html += "</tr>"
        for r in range(rows):
            html += '<tr><th style="padding:4px 12px 4px 4px;text-align:left;font-weight:600">%s</th>' % escape(
                row_labels[r])
            for c in range(columns):
                html += ('<td style="padding:5px 12px;min-width:2.2em;text-align:center;'
                         'font-variant-numeric:tabular-nums;%s%s">%s</td>'
                         % (edge, _p1_tint(matrix[r][c]), escape(entries[r][c])))
            html += "</tr>"
        html += "</table>"
        html += '<div style="font-size:13px;opacity:0.75;margin:0 0 8px 2px">%s</div>' % escape(size)

        shell = None
        if _p1_get_ipython is not None:
            shell = _p1_get_ipython()
        if _p1_display is None or shell is None:
            print(text)
        else:
            _p1_display({"text/plain": text, "text/html": html}, raw=True)
    except Exception as error:
        print("The table could not be shown (%s). Run the setup cell of part 1 again, then this cell." % error)


def _p1_on_its_side(matrix):
    """The matrix turned on its side (its transpose): row j of the result is column j of matrix."""
    try:
        turned = []
        for j in range(len(matrix[0])):
            new_row = []
            for i in range(len(matrix)):
                new_row.append(matrix[i][j])
            turned.append(new_row)
        return turned
    except Exception:
        return None


# The checks

# Each test: the inputs, the expected answer, the call as you would type it, and why that is the answer.
_P1_TESTS = {
    "inner_product": [
        (([2, 1, -2], [2, 1, -1]), 7,
         '',
         ''),
        (([-2, 2, 2], [2, -2, 1]), -6,
         '',
         ''),
        (([2, 1, -2], [2, -2, 1]), 0,
         '',
         ''),
        (([1, -2, 0, 2], [1, 1, 0, -1]), -3,
         '',
         ''),
        (([2], [-2]), -4,
         '',
         ''),
    ],
    "scores_for_customer": [
        (([2, 1, -2], [[2, 1, -1], [2, -2, 1], [-2, 1, 2], [-2, -2, 0]]), [7, 0, -7, -6],
         'scores_for_customer(customer_matrix[0], film_matrix)',
         "Sara's score for each of the four films, in the order of film_matrix"),
        (([-2, 2, 2], [[2, 1, -1], [2, -2, 1], [-2, 1, 2], [-2, -2, 0]]), [-4, -6, 10, 0],
         'scores_for_customer(customer_matrix[1], film_matrix)',
         "Leo's score for each of the four films, in the order of film_matrix"),
        (([1, -2, 1], [[2, 1, -1], [2, -2, 1], [-2, 1, 2], [-2, -2, 0]]), [-1, 7, -2, 2],
         'scores_for_customer(customer_matrix[2], film_matrix)',
         "Maya's score for each of the four films, in the order of film_matrix"),
        (([2, 1, -2], [[-2, -2, 0], [2, 1, -1]]), [-6, 7],
         'scores_for_customer([2, 1, -2], [[-2, -2, 0], [2, 1, -1]])',
         'Sara with two films only, Interstellar first and The Intouchables second: one score for each, in that order'),
        (([-2, 2, 2], [[-2, 1, 2]]), [10],
         'scores_for_customer([-2, 2, 2], [[-2, 1, 2]])',
         'With one film the answer is a list that holds one score'),
        (([1, -2, 0, 2], [[1, 1, 0, -1], [0, -2, 2, 1], [2, 0, -1, 1], [-1, 1, 1, 0], [1, 2, -2, 2], [0, -1, 1, -2]]), [-3, 6, 4, -3, 1, -2],
         'scores_for_customer(big_a[0], big_b)',
         'big_a and big_b are the bigger pair of scene 4 (you meet them at the end of part 1): a row of 4 numbers and a matrix of 6 rows give 6 scores'),
    ],
    "matrix_multiply": [
        (([[2, 1, -2], [-2, 2, 2], [1, -2, 1]], [[2, 1, -1], [2, -2, 1], [-2, 1, 2], [-2, -2, 0]]), [[7, 0, -7, -6], [-4, -6, 10, 0], [-1, 7, -2, 2]],
         'matrix_multiply(customer_matrix, film_matrix)',
         "The Matchmaker's table: one row per customer, one column per film"),
        (([[2, 1, -2]], [[2, 1, -1], [2, -2, 1], [-2, 1, 2], [-2, -2, 0]]), [[7, 0, -7, -6]],
         'matrix_multiply([[2, 1, -2]], film_matrix)',
         "A matrix with one row, Sara's, times the films: a table with one row"),
        (([[2, 1, -1], [2, -2, 1], [-2, 1, 2], [-2, -2, 0]], [[2, 1, -2], [-2, 2, 2], [1, -2, 1]]), [[7, -4, -1], [0, -6, 7], [-7, 10, -2], [-6, 0, 2]],
         'matrix_multiply(film_matrix, customer_matrix)',
         'The other way round: one row per film, one column per customer'),
        (([[3]], [[-2]]), [[-6]],
         'matrix_multiply([[3]], [[-2]])',
         'Two matrices of one row and one column: a table of one row that holds one number'),
        (([[2, 1, -2], [-2, 2, 2], [1, -2, 1]], [[2, 1, -2], [-2, 2, 2], [1, -2, 1]]), [[9, -6, -2], [-6, 12, -4], [-2, -4, 6]],
         'matrix_multiply(customer_matrix, customer_matrix)',
         'A matrix times itself: every customer with every customer'),
        (([[1, -2, 0, 2], [2, 1, -1, 0], [-1, 0, 2, 1], [0, 2, 1, -2], [2, -1, 1, 1]], [[1, 1, 0, -1], [0, -2, 2, 1], [2, 0, -1, 1], [-1, 1, 1, 0], [1, 2, -2, 2], [0, -1, 1, -2]]), [[-3, 6, 4, -3, 1, -2], [3, -4, 5, -2, 6, -2], [-2, 5, -3, 3, -3, 0], [4, -4, -3, 3, -2, 3], [0, 5, 4, -2, 0, 0]],
         'matrix_multiply(big_a, big_b)',
         'The bigger pair of scene 4 (you meet it at the end of part 1)'),
    ],
    "can_multiply": [
        (([[2, 1, -2], [-2, 2, 2], [1, -2, 1]], [[2, 1, -1], [2, -2, 1], [-2, 1, 2], [-2, -2, 0]]), True,
         'can_multiply(customer_matrix, film_matrix)',
         'Both have 3 columns, one per axis. Their numbers of rows, 3 and 4, play no part in the rule'),
        (([[1, -2, 0, 2], [2, 1, -1, 0], [-1, 0, 2, 1], [0, 2, 1, -2], [2, -1, 1, 1]], [[1, 1, 0, -1], [0, -2, 2, 1], [2, 0, -1, 1], [-1, 1, 1, 0], [1, 2, -2, 2], [0, -1, 1, -2]]), True,
         'can_multiply(big_a, big_b)',
         'Both have 4 columns: the bigger pair of scene 4, which you meet at the end of part 1'),
        (([[1, 2, 3], [4, 5, 6], [7, 8, 9]], [[1, 0], [0, 1], [1, 1], [2, 0]]), False,
         'can_multiply([[1, 2, 3], [4, 5, 6], [7, 8, 9]], [[1, 0], [0, 1], [1, 1], [2, 0]])',
         'Rows of 3 numbers and rows of 2 numbers: the third number has no partner'),
        (([[1, 2]], [[3, 4], [5, 6], [7, 8]]), True,
         'can_multiply([[1, 2]], [[3, 4], [5, 6], [7, 8]])',
         'One row and three rows, 2 columns each'),
        (([[1, 2, 3], [4, 5, 6]], [[1, 2], [3, 4]]), False,
         'can_multiply([[1, 2, 3], [4, 5, 6]], [[1, 2], [3, 4]])',
         'Two rows each, with 3 columns and 2 columns'),
        (([[2, 1, -2], [-2, 2, 2], [1, -2, 1]], [[1, 1, 0, -1], [0, -2, 2, 1], [2, 0, -1, 1], [-1, 1, 1, 0], [1, 2, -2, 2], [0, -1, 1, -2]]), False,
         'can_multiply(customer_matrix, big_b)',
         '3 columns and 4 columns'),
        (([[1, 2, 0]], [[1, 0], [0, 1], [2, 2]]), False,
         'can_multiply([[1, 2, 0]], [[1, 0], [0, 1], [2, 2]])',
         '3 columns and 2 columns, although the second matrix has 3 rows'),
    ],
}

_P1_HINTS = {'scores_for_customer': 'in the loop, score = inner_product(customer_row, film_matrix[j]), then scores.append(score).', 'matrix_multiply': 'row i of the product is scores_for_customer(matrix_a[i], matrix_b): the inner products of matrix_a[i] with every row of matrix_b.', 'can_multiply': 'columns_b = len(matrix_b[0]); then, if columns_a == columns_b, return True, else return False.'}
_P1_SUCCESS = {'one_score': "Task A works: Leo's row [-2, 2, 2] and the row of Men in Black [-2, 1, 2] give (-2)*(-2) + 2*1 + 2*2 = 4 + 2 + 4 = 10.", 'scores_for_customer': "Sara's scores for the four films are [7, 0, -7, -6].", 'matrix_multiply': "It gives the Matchmaker's table of scores, 3 rows and 4 columns, and the bigger product of scene 4, 5 rows and 6 columns.", 'can_multiply': 'True for the customers and the films, 3 columns each; False for 3 columns and 2 columns.', 'big_product': 'big_product has 5 rows and 6 columns and the same 30 numbers as scene 4 of the game.'}
_P1_COUNTING = {'customer_matrix': "Python counts from 0: customer_matrix[0] is Sara's row, customer_matrix[1] Leo's and customer_matrix[2] Maya's.", 'film_matrix': 'Python counts from 0: film_matrix[0] is the row of The Intouchables, film_matrix[1] of Apollo 13, film_matrix[2] of Men in Black and film_matrix[3] of Interstellar.'}


def _p1_call(fn, inputs):
    """Run a participant's function on fresh copies of the inputs, with its prints silenced.
    Gives back (answer, error), where error is None when the call went through. Never raises."""
    import contextlib, copy, io
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            answer = fn(*copy.deepcopy(inputs))
        return answer, None
    except Exception as error:
        return None, "%s: %s" % (type(error).__name__, error)


def _p1_right(name, answer, expected):
    """True when answer is the expected one; can_multiply must answer True or False themselves."""
    try:
        if name == "can_multiply":
            return isinstance(answer, bool) and answer == expected
        return bool(answer == expected)
    except Exception:
        return False


def _p1_first_failure(name, fn):
    """The first test of name that fn fails, or None when fn passes them all."""
    for inputs, expected, call, why in _P1_TESTS[name]:
        answer, error = _p1_call(fn, inputs)
        if error is not None or not _p1_right(name, answer, expected):
            return {"inputs": inputs, "answer": answer, "error": error, "expected": expected, "call": call,
                    "why": why}
    return None


def _p1_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    # A gap such as "x = ..." stays in the compiled function, so no source is needed; a gap that stands on
    # its own line is only visible in the source, which Jupyter and Colab keep for every cell
    code = getattr(fn, "__code__", None)
    if code is not None and Ellipsis in code.co_consts:
        return True
    import inspect
    try:
        lines = inspect.getsource(fn).splitlines()
    except Exception:
        return False
    for line in lines:
        if "..." in line.split("#")[0]:
            return True
    return False


def _p1_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _p1_rows_of(value):
    """True when value is a list of lists."""
    return isinstance(value, list) and len(value) > 0 and isinstance(value[0], list)


def _p1_explain_row(failure):
    """(what went wrong, hint) for scores_for_customer."""
    answer, expected = failure["answer"], failure["expected"]
    customer_row, film_matrix = failure["inputs"]
    error = failure["error"]
    if error is not None:
        if "IndexError" in error:
            return (None, "an index went past the end of a list. In the loop, j counts the films: "
                          "film_matrix[j] is the row of film j, and customer_row is the customer's whole row.")
        if "TypeError" in error:
            return (None, "inner_product takes two rows, each a list of numbers: customer_row and "
                          "film_matrix[j], the row of one film. film_matrix alone is the whole matrix.")
        return (None, _P1_HINTS["scores_for_customer"])
    if answer is None:
        return (None, "end the function with return scores.")
    if not isinstance(answer, list):
        return ("gave %s, expected the list %r" % (_p1_short(answer), expected),
                "return the list scores, with one score per film. A single number is the score of one film only.")
    if len(answer) == 0:
        return ("gave an empty list, expected %r" % expected,
                "scores is still empty: in each round of the loop, put the score at the end of the list with "
                "scores.append(score).")
    for x in answer:
        if isinstance(x, (list, tuple)):
            return ("gave a list of lists, %s, expected %r" % (_p1_short(answer), expected),
                    "each entry of scores is one number, a score. Did you append the row of a film, "
                    "film_matrix[j], in place of its score?")
    if len(answer) != len(expected):
        return ("gave %s, %s, expected %d, %r" % (_p1_count(len(answer), "score"), _p1_short(answer),
                                                  len(expected), expected),
                "one score per row of film_matrix: append exactly one score in each round of the loop, inside the "
                "loop.")
    if len(answer) > 1 and answer.count(answer[0]) == len(answer) and expected.count(expected[0]) != len(expected):
        return ("gave %r, expected %r" % (answer, expected),
                "every score is the same: in round j of the loop, use the row of film j, film_matrix[j].")
    sara = []
    for film_row in film_matrix:
        sara.append(_p1_inner(_P1_CUSTOMER_MATRIX[0], film_row))
    if answer == sara and customer_row != _P1_CUSTOMER_MATRIX[0]:
        return ("gave %r, expected %r" % (answer, expected),
                "these are %s's scores: use customer_row, the row that the function gets, in place of "
                "customer_matrix[0]." % _P1_CUSTOMER_NAMES[0])
    return ("gave %r, expected %r" % (answer, expected), _P1_HINTS["scores_for_customer"])


def _p1_explain_product(answer, matrix_a, matrix_b, expected):
    """(what went wrong, hint) for a product that differs from the expected one. answer is not None."""
    rows, columns = len(expected), len(expected[0])
    wanted = "%s, one for each row of matrix_a, and %s, one for each row of matrix_b" % (
        _p1_count(rows, "row"), _p1_count(columns, "column"))
    if not isinstance(answer, list):
        return ("gave %s; expected a table of %s" % (_p1_short(answer), wanted),
                "return product, the list of rows.")
    if len(answer) == 0:
        return ("gave an empty list; expected a table of %s" % wanted,
                "product is still empty: append each new_row to product, inside the loop.")
    flat = True
    for row in answer:
        if not _p1_is_number(row):
            flat = False
    if flat:
        return ("gave a list of %s, %s; expected a table of %s" % (_p1_count(len(answer), "number"),
                                                                   _p1_short(answer), wanted),
                "the product is a list of rows. new_row is a whole row, the scores of matrix_a[i] with every "
                "row of matrix_b, and product.append(new_row) adds it to the table.")
    for r in range(len(answer)):
        row = answer[r]
        if row is None:
            return ("gave a table whose row %d is None" % r,
                    "new_row is None: does scores_for_customer end with return scores?")
        if not isinstance(row, list):
            return ("gave a table whose row %d is %s, a single value" % (r, _p1_short(row)),
                    "each row of the product is a list of numbers, one for each row of matrix_b.")
    for row in answer:
        for x in row:
            if isinstance(x, (list, tuple)):
                return ("gave a table whose entries are lists, such as %s" % _p1_short(x),
                        "each entry of the product is one number, an inner product. Put numbers into new_row: "
                        "appending a whole list puts a list inside the row.")
    # The Matchmaker's table again, although the function got other matrices: it reads the notebook's lists
    if (matrix_a, matrix_b) != (_P1_CUSTOMER_MATRIX, _P1_FILM_MATRIX):
        matchmaker = []
        for customer_row in _P1_CUSTOMER_MATRIX:
            scores = []
            for film_row in _P1_FILM_MATRIX:
                scores.append(_p1_inner(customer_row, film_row))
            matchmaker.append(scores)
        if answer == matchmaker:
            return ("gave the Matchmaker's table of scores again; expected a table of %s" % wanted,
                    "use matrix_a and matrix_b, the two matrices that the function gets, in place of "
                    "customer_matrix and film_matrix.")
    # Row i paired with the wrong matrix: scores_for_customer(matrix_b[i], matrix_a)
    if len(matrix_b) >= len(matrix_a) and len(matrix_a) != len(matrix_b):
        turned_call = []
        for i in range(len(matrix_a)):
            scores = []
            for row_a in matrix_a:
                scores.append(_p1_inner(matrix_b[i], row_a))
            turned_call.append(scores)
        if answer == turned_call:
            return ("gave a table of %s and %s; expected %s" % (_p1_count(len(answer), "row"),
                                                                _p1_count(len(answer[0]), "column"), wanted),
                    "the two matrices changed places: row i of the product is scores_for_customer(matrix_a[i], "
                    "matrix_b), with a row of matrix_a first and the whole matrix_b second.")
    lengths = []
    for row in answer:
        lengths.append(len(row))
    growing = []
    for r in range(len(answer)):
        growing.append(columns * (r + 1))
    one_list = True
    for row in answer:
        if row is not answer[0]:
            one_list = False
    if len(answer) > 1 and (lengths == growing or (one_list and lengths[0] != columns)):
        return ("gave rows of %s numbers; expected %s" % (_p1_and(lengths), wanted),
                "new_row keeps growing: start a new, empty row, new_row = [], for each row of matrix_a, inside "
                "the outer loop.")
    if min(lengths) != max(lengths):
        return ("gave rows of different lengths, %s; expected %s" % (_p1_and(lengths), wanted),
                "every row of the product has one number for each row of matrix_b.")
    got_rows, got_columns = len(answer), lengths[0]
    if (got_rows, got_columns) != (rows, columns):
        got = "a table of %s and %s" % (_p1_count(got_rows, "row"), _p1_count(got_columns, "column"))
        if (got_rows, got_columns) == (columns, rows):
            return ("gave %s; expected %s" % (got, wanted),
                    "rows and columns are swapped. The outer loop goes over the rows of matrix_a, and each new "
                    "row holds one inner product for each row of matrix_b.")
        if got_rows != rows:
            return ("gave %s; expected %s" % (got, wanted),
                    "the product has one row for each row of matrix_a: append one new_row in each round of the "
                    "loop over the rows of matrix_a.")
        return ("gave %s; expected %s" % (got, wanted),
                "each row of the product holds one number for each row of matrix_b: the inner products of "
                "matrix_a[i] with matrix_b[0], matrix_b[1] and so on.")
    # The right size, some wrong numbers
    flat_rows = True
    for r in range(rows):
        if answer[r].count(answer[r][0]) != columns:
            flat_rows = False
    expected_flat = True
    for r in range(rows):
        if expected[r].count(expected[r][0]) != columns:
            expected_flat = False
    if columns > 1 and flat_rows and not expected_flat:
        return ("has the right size, %s and %s, but every number in a row is the same: row 0 is %r, expected %r"
                % (_p1_count(rows, "row"), _p1_count(columns, "column"), answer[0], expected[0]),
                "the entry in row i and column j pairs matrix_a[i] with matrix_b[j], a different row of matrix_b "
                "for each column. Check which index goes with which matrix.")
    for i in range(rows):
        for j in range(columns):
            if answer[i][j] != expected[i][j]:
                return ("has the right size, %s and %s, but the entry in row %d and column %d is %s, expected %r"
                        % (_p1_count(rows, "row"), _p1_count(columns, "column"), i, j, _p1_short(answer[i][j]),
                           expected[i][j]),
                        "the entry in row %d and column %d is the inner product of matrix_a[%d], %r, with "
                        "matrix_b[%d], %r." % (i, j, i, matrix_a[i], j, matrix_b[j]))
    return ("gave %s" % _p1_short(answer), _P1_HINTS["matrix_multiply"])


def _p1_explain_columns(fn, failure):
    """(what went wrong, hint) for can_multiply, from the pattern of its answers on all the tests."""
    answer, error = failure["answer"], failure["error"]
    if error is not None:
        if "IndexError" in error:
            return (None, "use the first row of each matrix, matrix_a[0] and matrix_b[0]: every matrix has one, "
                          "even a matrix with a single row.")
        return (None, "len gives the length of a list: len(matrix_a[0]) is the number of columns of matrix_a.")
    if answer is None:
        return (None, "end with a return line: if columns_a == columns_b, return True, else return False.")
    if not isinstance(answer, bool):
        if answer in ("True", "False"):
            return ("gave the text %r, expected %r" % (answer, failure["expected"]),
                    "return True or False without quotes: they are Python's own values for yes and no.")
        return ("gave %s, expected %r" % (_p1_short(answer), failure["expected"]),
                "return True or False: compare the two numbers of columns with == inside an if.")
    answers, expected, rows, columns_rows, rows_columns = [], [], [], [], []
    for inputs, wanted, call, why in _P1_TESTS["can_multiply"]:
        matrix_a, matrix_b = inputs
        answers.append(_p1_call(fn, inputs)[0])
        expected.append(wanted)
        rows.append(len(matrix_a) == len(matrix_b))
        columns_rows.append(len(matrix_a[0]) == len(matrix_b))
        rows_columns.append(len(matrix_a) == len(matrix_b[0]))
    what = "gave %r, expected %r" % (answer, failure["expected"])
    if answers == rows:
        return (what, "you compared the numbers of rows, len(matrix_a) and len(matrix_b). The rule is about the "
                      "columns: compare the lengths of the rows, len(matrix_a[0]) and len(matrix_b[0]).")
    if answers == columns_rows:
        return (what, "you compared the columns of matrix_a with the number of rows of matrix_b, len(matrix_b). "
                      "Compare the columns of both: len(matrix_a[0]) and len(matrix_b[0]).")
    if answers == rows_columns:
        return (what, "you compared the number of rows of matrix_a, len(matrix_a), with the columns of matrix_b. "
                      "Compare the columns of both: len(matrix_a[0]) and len(matrix_b[0]).")
    if answers == [True] * len(answers):
        return (what, "your function says True every time, also for a pair that cannot be multiplied. Return "
                      "False when columns_a and columns_b differ (else: return False).")
    if answers == [False] * len(answers):
        return (what, "your function never says True. Return True when columns_a == columns_b.")
    flipped = []
    for x in expected:
        flipped.append(not x)
    if answers == flipped:
        return (what, "True and False are swapped: return True when the numbers of columns are equal.")
    return (what, _P1_HINTS["can_multiply"])


def _p1_explain(name, fn, failure):
    """(what went wrong, hint) for the first test that fn fails."""
    if name == "scores_for_customer":
        return _p1_explain_row(failure)
    if name == "can_multiply":
        return _p1_explain_columns(fn, failure)
    if failure["error"] is not None:
        error = failure["error"]
        if "IndexError" in error:
            return (None, "an index went past the end of a matrix. i counts the rows of matrix_a, so row i is "
                          "matrix_a[i]; inside, j counts the rows of matrix_b: pair matrix_a[i] with matrix_b[j].")
        if "TypeError" in error:
            return (None, "inner_product takes two rows, such as matrix_a[i] and matrix_b[j]; "
                          "scores_for_customer takes one row and a whole matrix, such as matrix_a[i] and matrix_b.")
        return (None, _P1_HINTS["matrix_multiply"])
    if failure["answer"] is None:
        return (None, "end the function with return product.")
    matrix_a, matrix_b = failure["inputs"]
    return _p1_explain_product(failure["answer"], matrix_a, matrix_b, failure["expected"])


def _p1_check(name, fn, needs):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    failure = _p1_first_failure(name, fn)
    if failure is None:
        print("✅ %s works on %d tests. %s" % (name, len(_P1_TESTS[name]), _P1_SUCCESS[name]))
        return
    not_finished = ("⏳ %s is not finished yet: replace each ... (Python calls it Ellipsis) with your code, "
                    "run its cell, then run this check again." % name)
    if _p1_has_gap(fn):
        print(not_finished)
        return
    for other, place in needs:
        if not _p1_uses(fn, other):
            continue
        other_fn = globals().get(other)
        if other_fn is None or _p1_first_failure(other, other_fn) is not None:
            if other == "inner_product":
                print("⏳ %s uses inner_product, which does not work any more. Run the cell of inner_product "
                      "again (it is given, so undo any change to it), then run this check again." % name)
            else:
                print("⏳ %s uses your %s from %s, which does not work yet. Get the check of %s to ✅ first, "
                      "then run this check again." % (name, other, place, place))
            return
    if failure["answer"] is Ellipsis or "ellipsis" in str(failure["error"]).lower():
        print(not_finished)
        return
    what, hint = _p1_explain(name, fn, failure)
    if failure["error"] is not None:
        print("❌ %s stopped with an error: %s" % (failure["call"], failure["error"]))
    elif failure["answer"] is None:
        print("❌ %s gave no answer (None). Does the function end with a return line?" % failure["call"])
    else:
        print("❌ %s %s. %s." % (failure["call"], what, failure["why"]))
    print("   💡 Hint: " + hint)


def check_scores_for_customer(fn):
    """Task B: tests scores_for_customer on the three customers, on fewer films and on the bigger pair."""
    try:
        _p1_check("scores_for_customer", fn, [("inner_product", "the given cell")])
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 1 again, then this cell." % error)


def check_matrix_multiply(fn):
    """Task C: tests matrix_multiply on six pairs of matrices, from 1 x 1 to the bigger pair of scene 4."""
    try:
        _p1_check("matrix_multiply", fn, [("inner_product", "the given cell"),
                                          ("scores_for_customer", "task B")])
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 1 again, then this cell." % error)


def check_can_multiply(fn):
    """Task D: tests can_multiply on seven pairs of matrices, some that fit and some that do not."""
    try:
        _p1_check("can_multiply", fn, [])
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 1 again, then this cell." % error)


def _p1_which_row(value, matrix):
    """The index of value among the rows of matrix, or -1."""
    for i in range(len(matrix)):
        try:
            if value == matrix[i]:
                return i
        except Exception:
            pass
    return -1


def _p1_row_problem(value, variable, matrix_name, matrix, names, wanted, other_name, other_matrix):
    """None when value is row number wanted of matrix, else (what went wrong, hint), for task A."""
    if _p1_which_row(value, [matrix[wanted]]) == 0:
        return None
    k = _p1_which_row(value, matrix)
    if k >= 0:
        return ("%s is %s, the row of %s, %s[%d]" % (variable, _p1_short(value), names[k], matrix_name, k),
                _P1_COUNTING[matrix_name])
    if _p1_which_row(value, other_matrix) >= 0:
        return ("%s is %s, a row of %s" % (variable, _p1_short(value), other_name),
                "the row of %s is in %s." % (names[wanted], matrix_name))
    if _p1_is_number(value):
        return ("%s is %s, a single number" % (variable, _p1_short(value)),
                "a row is a whole list of numbers, and %s[%d] picks one. A second index, as in %s[%d][0], "
                "would pick one number out of that row." % (matrix_name, wanted, matrix_name, wanted))
    if _p1_rows_of(value):
        return ("%s is a whole matrix" % variable,
                "pick one row out of it with an index in square brackets, such as %s[0] for the first row."
                % matrix_name)
    return ("%s is %s, and the row of %s is a list of %d numbers" % (variable, _p1_short(value), names[wanted],
                                                                     len(matrix[wanted])),
            _P1_COUNTING[matrix_name])


def _p1_check_one(leo_row, film_row, score):
    gaps = []
    for variable, value in (("leo_row", leo_row), ("men_in_black_row", film_row), ("leo_and_men_in_black", score)):
        if value is Ellipsis:
            gaps.append(variable)
    if gaps:
        verb = "holds"
        if len(gaps) > 1:
            verb = "hold"
        print("⏳ Task A is not finished yet: %s still %s ... (Python calls it Ellipsis). Replace each ... with "
              "your code, run the cell, then run this check again." % (_p1_and(gaps), verb))
        return
    c, f = _P1_ONE
    problem = _p1_row_problem(leo_row, "leo_row", "customer_matrix", _P1_CUSTOMER_MATRIX, _P1_CUSTOMER_NAMES, c,
                              "film_matrix", _P1_FILM_MATRIX)
    if problem is None:
        problem = _p1_row_problem(film_row, "men_in_black_row", "film_matrix", _P1_FILM_MATRIX, _P1_FILM_NAMES, f,
                                  "customer_matrix", _P1_CUSTOMER_MATRIX)
    if problem is None:
        expected = _p1_inner(_P1_CUSTOMER_MATRIX[c], _P1_FILM_MATRIX[f])
        if not _p1_is_number(score):
            problem = ("leo_and_men_in_black is %s, and a score is one number" % _p1_short(score),
                       "leo_and_men_in_black = inner_product(leo_row, men_in_black_row).")
        elif score != expected:
            fn = globals().get("inner_product")
            if fn is None or _p1_first_failure("inner_product", fn) is not None:
                print("⏳ inner_product does not work any more. Run its cell again (it is given, so undo any change "
                      "to it), then the cell of task A, then this check.")
                return
            problem = ("leo_and_men_in_black is %s, expected %d" % (_p1_short(score), expected),
                       "let inner_product do the work: leo_and_men_in_black = inner_product(leo_row, "
                       "men_in_black_row).")
    if problem is None:
        print("✅ " + _P1_SUCCESS["one_score"])
        return
    print("❌ %s." % problem[0])
    print("   💡 Hint: " + problem[1])


def check_one_score(leo_row, men_in_black_row, leo_and_men_in_black):
    """Task A: Leo's row, the row of Men in Black, and their score."""
    try:
        _p1_check_one(leo_row, men_in_black_row, leo_and_men_in_black)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 1 again, then this cell." % error)


def _p1_check_big(product):
    for name, mine in (("big_a", _P1_BIG_A), ("big_b", _P1_BIG_B)):
        if name in globals() and globals()[name] != mine:
            print("❌ %s is no longer the matrix of scene 4. Run the cell with big_a and big_b again, then the "
                  "cells below it." % name)
            return
    problem = _p1_not_a_table(product)
    if product is None or product is Ellipsis or (problem is not None and problem[0] == "wait"):
        print("⏳ big_product is not ready yet: your matrix_multiply of task C is not finished. Get its check to "
              "✅, then run the cells of the bigger pair again.")
        return
    if product == _P1_BIG_PRODUCT:
        print("✅ " + _P1_SUCCESS["big_product"])
        return
    what, hint = _p1_explain_product(product, _P1_BIG_A, _P1_BIG_B, _P1_BIG_PRODUCT)
    print("❌ big_product %s." % what.replace("gave ", "is ", 1))
    print("   💡 Hint: " + hint + " The check of task C tells you more.")


def check_big_product(big_product):
    """The bigger pair: compares big_product with the product of scene 4 of the game."""
    try:
        _p1_check_big(big_product)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 1 again, then this cell." % error)


print("✅ Setup of part 1 complete.")

## The two matrices of the game

The Matchmaker's numbers as Python lists. A matrix is a list of rows. `customer_matrix` has one row per customer,
and each row holds three numbers, one per axis, in the order of `axes`. `film_matrix` has one row per film, on the
same three axes. Python counts from 0: `customer_matrix[0]` is Sara's row, `[2, 1, -2]`, and `film_matrix[0]` is the row
of *The Intouchables*, `[2, 1, -1]`. The customers, their profiles and the critic are invented for the lesson.

In [ ]:
# The Matchmaker's data, from section 1 of the game (invented for the lesson)
axes = ["real", "comedy", "action"]
customer_names = ["Sara", "Leo", "Maya"]
film_names = ["The Intouchables", "Apollo 13", "Men in Black", "Interstellar"]

# The customers' profiles: one row per customer, one number per axis
customer_matrix = [
    [ 2,  1, -2],   # Sara
    [-2,  2,  2],   # Leo
    [ 1, -2,  1],   # Maya
]

# The critic's answers: one row per film, on the same three axes
film_matrix = [
    [ 2,  1, -1],   # The Intouchables (2011)
    [ 2, -2,  1],   # Apollo 13 (1995)
    [-2,  1,  2],   # Men in Black (1997)
    [-2, -2,  0],   # Interstellar (2014)
]

In [ ]:
print("customer_matrix, the profiles of the customers:")
show_matrix(customer_matrix, customer_names, axes)
print("film_matrix, the answers of the critic:")
show_matrix(film_matrix, film_names, axes)

## The inner product, given

You wrote this function yourself in the video store notebook: multiply two rows position by position, then add
up the products. Here it is, ready to use. Run its cell, and the next one, which uses it once.

In [ ]:
# Given: the inner product of two rows, the function you wrote in the video store notebook
def inner_product(row_a, row_b):
    # row_a and row_b are two lists of numbers of the same length.
    # Multiply them position by position, then add up the products.
    total = 0
    for i in range(len(row_a)):
        total = total + row_a[i] * row_b[i]
    return total

In [ ]:
# Sara is row 0 of customer_matrix, and The Intouchables is row 0 of film_matrix
print("Sara and The Intouchables:", inner_product(customer_matrix[0], film_matrix[0]))

**Reading the result.** 7, as in scene 2 of the game. Sara's row `[2, 1, -2]` sits on top of the row of *The Intouchables*, `[2, 1, -1]`: the
products under the three axes are 4, 1 and 2, and their sum is 4 + 1 + 2 = 7. They agree on all three axes. In Python:
`2*2 + 1*1 + (-2)*(-1) = 4 + 1 + 2 = 7`.

## 🎯 Task A. One score

What does Leo think of *Men in Black*? Pick the two rows out of the matrices by their index, then let `inner_product`
compute the score. Remember that Python counts from 0: Sara is customer 0, and *The Intouchables* is film 0. In the cell
below, replace each `...` with your code and run it.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# Python counts the rows of a matrix from 0: customer_matrix[0] is the first row, Sara's.
# Leo is the second customer. Which number picks his row?
# Men in Black is the third film in film_matrix. Which number picks its row?

In [ ]:
#@title 💡 Hint 2 for task A: the shape of the code (double-click to open) { display-mode: "form" }
# leo_row = customer_matrix[1]
# men_in_black_row = film_matrix[...]                  # the third film: count 0, 1, 2
# leo_and_men_in_black = inner_product(leo_row, men_in_black_row)

In [ ]:
# 🎯 YOUR TURN, task A: one score, Leo and Men in Black
leo_row = ...                # 🎯 YOUR TURN: Leo's row of customer_matrix, picked by its index
men_in_black_row = ...       # 🎯 YOUR TURN: the row of Men in Black in film_matrix
leo_and_men_in_black = ...   # 🎯 YOUR TURN: the inner product of the two rows

print("Leo's row:           ", leo_row)
print("Men in Black's row:  ", men_in_black_row)
print("Leo and Men in Black:", leo_and_men_in_black)

In [ ]:
check_one_score(leo_row, men_in_black_row, leo_and_men_in_black)

## 🎯 Task B. One row

A customer's row of the table of scores holds four scores, one for each film. Write
`scores_for_customer(customer_row, film_matrix)`. It gets the row of one customer and the matrix of the films, and
returns a list with the customer's score for every film, in the order of the films. For Sara,
`scores_for_customer(customer_matrix[0], film_matrix)` gives `[7, 0, -7, -6]`.

The loop is given: `for j in range(len(film_matrix))` runs once for each film, with j equal to 0, 1, 2 and 3, so
`film_matrix[j]` is the row of film j. A list grows with `append`: after `scores = []` and `scores.append(7)`,
`scores` is `[7]`.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# The loop runs once for each film: j is 0, then 1, 2 and 3, and film_matrix[j] is the row of film j.
# In each round, the score is the inner product of customer_row with film_matrix[j].
# Then put that score at the end of the list: scores.append(score).

In [ ]:
#@title 💡 Hint 2 for task B: the shape of the code (double-click to open) { display-mode: "form" }
#     for j in range(len(film_matrix)):
#         score = inner_product(customer_row, film_matrix[j])
#         ...                                   # put score at the end of the list scores

In [ ]:
# 🎯 YOUR TURN, task B: one customer's scores for every film
def scores_for_customer(customer_row, film_matrix):
    # customer_row is the row of one customer; film_matrix has one row per film.
    # Return a list with the customer's score for every film, in the order of the films.
    scores = []
    for j in range(len(film_matrix)):
        # film_matrix[j] is the row of film number j
        score = ...   # 🎯 YOUR TURN: the customer's score for film j (use inner_product)
        ...           # 🎯 YOUR TURN: put score at the end of the list scores
    return scores

In [ ]:
check_scores_for_customer(scores_for_customer)

In [ ]:
# Sara's scores, shown as a table with one row: a list that holds the list of her scores
sara_scores = scores_for_customer(customer_matrix[0], film_matrix)
show_matrix([sara_scores], ["Sara"], film_names)

## 🎯 Task C. The whole table

The table of scores has one row per customer, and each row is what your `scores_for_customer` computes. So the
whole table is a loop around your loop: for every row of the first matrix, its scores with every row of the
second. That is **matrix multiplication**, as in scene 3 of the game. Two matrices with the same number of
columns are multiplied by taking the inner product of every row of the first with every row of the second. The
product has one row for each row of the first matrix and one column for each row of the second: its entry in row
i and column j is the inner product of row i of the first with row j of the second.

Write `matrix_multiply(matrix_a, matrix_b)`. It returns the product as a list of rows. You may assume that the
two matrices have the same number of columns. `matrix_multiply(customer_matrix, film_matrix)` should give the
Matchmaker's table of scores, 3 rows and 4 columns.

💭 **Think first.** How many inner products does the whole table take?

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# Read matrix_a as the customers and matrix_b as the films.
# Row i of the product is then customer i's scores for every film:
# the inner products of matrix_a[i] with every row of matrix_b.
# In task B you wrote a function that computes exactly one such row.

In [ ]:
#@title 💡 Hint 2 for task C (double-click to open) { display-mode: "form" }
# scores_for_customer(customer_row, film_matrix) from task B makes one row of the product.
# Here the customer's row is matrix_a[i], and the matrix of the films is matrix_b.

In [ ]:
#@title 💡 Hint 3 for task C: the shape of the code (double-click to open) { display-mode: "form" }
#     for i in range(len(matrix_a)):
#         new_row = scores_for_customer(matrix_a[i], ...)
#         product.append(new_row)
#
# You may also write the loop of task B again, inside this loop:
#         new_row = []
#         for j in range(len(matrix_b)):
#             new_row.append(inner_product(..., matrix_b[j]))
#         product.append(new_row)

In [ ]:
# 🎯 YOUR TURN, task C: the whole table, matrix multiplication
def matrix_multiply(matrix_a, matrix_b):
    # matrix_a and matrix_b have the same number of columns. The product has one row for
    # each row of matrix_a and one column for each row of matrix_b: its row i holds the
    # inner products of matrix_a[i] with every row of matrix_b.
    product = []
    for i in range(len(matrix_a)):
        new_row = ...   # 🎯 YOUR TURN: row i of the product, the scores of matrix_a[i]
        product.append(new_row)
    return product

In [ ]:
check_matrix_multiply(matrix_multiply)

In [ ]:
# The whole table of scores: the product of the customers' matrix and the films' matrix
show_matrix(matrix_multiply(customer_matrix, film_matrix), customer_names, film_names)

**Reading the result.** Twelve inner products, one for each cell: 3 customers times 4 films. Each customer has a
clear favourite: Sara *The Intouchables* (7), Leo *Men in Black* (10) and Maya *Apollo 13* (7). Two cells are 0: for Sara and *Apollo 13* the products 4, -2 and -2 cancel out, and so do 4, -4 and 0 for Leo and *Interstellar*.
These are the twelve scores of the game, and your `matrix_multiply` computed them all from the two matrices.
Inside it, `scores_for_customer` runs a loop over the films, so matrix multiplication is two loops, one inside
the other: every row of the first matrix with every row of the second.

## 🎯 Task D. The column rule

An inner product pairs two rows position by position, so the two rows must be equally long. That is why two
matrices can be multiplied only when they have the same number of columns: then each number in a row of the
first has a partner in every row of the second. The numbers of rows can differ, as with 3 customers and 4
films.

💭 **Think first.** Can a matrix of 3 rows and 3 columns be multiplied with a matrix of 4 rows and 2 columns?

Write `can_multiply(matrix_a, matrix_b)`. It returns `True` when the two matrices have the same number of
columns, and `False` otherwise. The number of columns of a matrix is the length of one of its rows:
`len(matrix_a[0])` is the length of the first row of `matrix_a`.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task D (double-click to open) { display-mode: "form" }
# The number of columns of a matrix is the length of one of its rows.
# columns_a = len(matrix_a[0]) measures the first row of matrix_a: do the same for matrix_b.
# Then compare the two numbers with == inside an if.

In [ ]:
#@title 💡 Hint 2 for task D: the shape of the code (double-click to open) { display-mode: "form" }
#     columns_b = len(matrix_b[0])
#     if columns_a == columns_b:
#         return True
#     else:
#         return ...

In [ ]:
# 🎯 YOUR TURN, task D: the column rule
def can_multiply(matrix_a, matrix_b):
    # Return True when matrix_a and matrix_b have the same number of columns, and False otherwise.
    columns_a = len(matrix_a[0])   # the number of columns of matrix_a: the length of its first row
    columns_b = ...                # 🎯 YOUR TURN: the number of columns of matrix_b
    # 🎯 YOUR TURN: if columns_a and columns_b are equal, return True; else return False
    ...

In [ ]:
check_can_multiply(can_multiply)

In [ ]:
# The think-first question: a matrix of 3 rows and 3 columns, and one of 4 rows and 2 columns
three_by_three = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
four_by_two = [[1, 0], [0, 1], [1, 1], [2, 0]]
print("Can they be multiplied?", can_multiply(three_by_three, four_by_two))

**Reading the result.** `False`: a row of the first matrix has 3 numbers and a row of the second only 2, so the
third number has no partner. The 4 rows of the second matrix play no part in the rule.

## A bigger product

Scene 4 of the game multiplied a bigger pair, `big_a` with 5 rows and 4 columns and `big_b` with 6 rows and 4 columns: plain invented numbers this
time, with no customers and no films. Without names, `show_matrix` numbers the rows and the columns from 0, the
way Python counts. The game numbered them from 1, so row 0 here is row 1 in the game.

In [ ]:
# The bigger pair of scene 4 of the game: plain invented numbers, 4 columns each
big_a = [
    [ 1, -2,  0,  2],
    [ 2,  1, -1,  0],
    [-1,  0,  2,  1],
    [ 0,  2,  1, -2],
    [ 2, -1,  1,  1],
]

big_b = [
    [ 1,  1,  0, -1],
    [ 0, -2,  2,  1],
    [ 2,  0, -1,  1],
    [-1,  1,  1,  0],
    [ 1,  2, -2,  2],
    [ 0, -1,  1, -2],
]

print("big_a:")
show_matrix(big_a)
print("big_b:")
show_matrix(big_b)

💭 **Think first.** Can `big_a` and `big_b` be multiplied? How many rows and columns will their product have? And
can `customer_matrix` be multiplied with `big_b`? Then run the cell, and the check below it.

In [ ]:
print("Can big_a and big_b be multiplied?", can_multiply(big_a, big_b))
print("Can customer_matrix and big_b be multiplied?", can_multiply(customer_matrix, big_b))

big_product = matrix_multiply(big_a, big_b)
show_matrix(big_product)

In [ ]:
check_big_product(big_product)

**Reading the result.** 5 rows, one for each row of `big_a`, and 6 columns, one for each row of `big_b`: 30
inner products of 4 products each, the same table as in scene 4 of the game. The entry in row 2 and column 3 (row 3
and column 4 in the game) is the inner product of `big_a[2]` and `big_b[3]`: `(-1)*(-1) + 0*1 + 2*1 + 1*0 = 1 + 0 + 2 + 0 = 3`. `customer_matrix` and `big_b` cannot be multiplied: their
rows have 3 and 4 numbers.

## Optional: the same product in textbooks

Textbooks, and the Python libraries you may meet later, write matrix multiplication with one change: they pair
the rows of the first matrix with the **columns** of the second. To get the Matchmaker's table that way, they
first turn the film matrix on its side, so that each film becomes a column. The turned matrix is called the
**transpose**. Every inner product stays the same, and so does the whole product. Their rule, "the columns of the
first match the rows of the second", is the column rule of this notebook, applied after the turn. The folded
cell below shows the film matrix turned on its side.

In [ ]:
#@title ▶ Optional: the film matrix turned on its side { display-mode: "form" }
print("film_matrix turned on its side: one column per film, one row per axis")
show_matrix(_p1_on_its_side(film_matrix), axes, film_names)

## Back to the game

That is part 1: your `matrix_multiply` fills the Matchmaker's whole table from the profiles. In real life nobody
fills in a profile, and the Matchmaker sees only the ratings. Section 2 of the game, "Only the ratings", starts
from the table and looks for the numbers behind it. Part 2 of this notebook comes after it.

# Part 2. Guess and correct

In section 2 of the game the Matchmaker lost the profiles: nobody fills one in, so it knows only the ratings
that its customers gave. The numbers behind the ratings cannot be computed from them directly. As Popper said,
we do not know: we can only guess. So we guess, measure how wrong the guess is, correct it, and go again.

In this part you teach the computer to do that on the corner of the game, Ida and *Paddington*:

- **Task A**: the squared loss of one prediction;
- **Task B**: the loss of a guess for the corner;
- **Task C**: one what-if, on one guessed number;
- **Task D**: the what-if loop, round after round, until the prediction matches the rating.

Then the loop runs on your own guess from the game, and on two guesses where it cannot move. Run the setup cell
of part 2 first.

If you come back after a break, Colab may have forgotten part 1. Then click on the setup cell below and choose
**Runtime > Run before** in Colab's menu: it runs every cell above it, part 1 included. Then run the setup
cell.

In [ ]:
#@title Setup of part 2 (run this first) { display-mode: "form" }
# The plumbing of part 2: print_move, which prints one move of the what-if loop in words, and the checks of
# tasks A to D. You do not need to read it. A check prints one short message and never stops the notebook.

# Part 2 uses two things from part 1: the given inner_product and show_matrix. When part 1 has not run in this
# session, this cell defines them, so that part 2 works on its own.
_p2_missing = []
if "inner_product" not in globals():
    _p2_missing.append("inner_product")

    def inner_product(row_a, row_b):
        # row_a and row_b are two lists of numbers of the same length.
        # Multiply them position by position, then add up the products.
        total = 0
        for i in range(len(row_a)):
            total = total + row_a[i] * row_b[i]
        return total

if "show_matrix" not in globals():
    _p2_missing.append("show_matrix")

    def show_matrix(matrix, row_names=None, column_names=None):
        """A plain stand-in for the show_matrix of part 1: the table as text, one line per row."""
        try:
            fine = isinstance(matrix, list) and len(matrix) > 0
            if fine:
                for row in matrix:
                    if not isinstance(row, list) or len(row) == 0 or len(row) != len(matrix[0]):
                        fine = False
                    else:
                        for x in row:
                            if x is Ellipsis or x is None or isinstance(x, (list, tuple)):
                                fine = False
            if not fine:
                print("⏳ Nothing to show yet: %s. Finish the 🎯 cell above, run it, then run this cell again."
                      % _p2_short(matrix))
                return
            rows, columns = len(matrix), len(matrix[0])
            if not isinstance(row_names, (list, tuple)) or len(row_names) != rows:
                row_names = [str(i) for i in range(rows)]
            if not isinstance(column_names, (list, tuple)) or len(column_names) != columns:
                column_names = [str(j) for j in range(columns)]
            width = max(len(str(name)) for name in row_names)
            widths = []
            for j in range(columns):
                w = len(str(column_names[j]))
                for i in range(rows):
                    w = max(w, len(str(matrix[i][j])))
                widths.append(w)
            print(" " * width + "".join("   " + str(column_names[j]).rjust(widths[j]) for j in range(columns)))
            for i in range(rows):
                print(str(row_names[i]).ljust(width)
                      + "".join("   " + str(matrix[i][j]).rjust(widths[j]) for j in range(columns)))
        except Exception as error:
            print("The table could not be shown (%s)." % error)

_P2_ROW_NAMES = ['Ida', 'Paddington']
_P2_POSITION_NAMES = ["first", "second"]
_P2_RATING = 2
_P2_ROUNDS = 100
_P2_ORDER = [(0, 0), (0, 1), (1, 0), (1, 1)]   # the rule's order: the customer's first and second, the film's
_P2_MAX_TRIES = 2000                           # the check of task D stops a loop that makes more what-ifs
_P2_ORDINALS = ["first", "second", "third", "fourth", "fifth", "sixth", "seventh", "eighth", "ninth", "tenth"]


def _p2_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = repr(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 80:
        text = text[:77] + "..."
    return text


def _p2_is_number(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)


def _p2_text(x):
    """A number as text: 16, or 0.25 for a decimal number."""
    if isinstance(x, float) and x == int(x):
        return "%d" % x
    if isinstance(x, float):
        return "%.4g" % x
    return str(x)


def _p2_name(row, position):
    """Ida's first number"""
    return "%s's %s number" % (_P2_ROW_NAMES[row], _P2_POSITION_NAMES[position])


def _p2_ordinal(k):
    """first, second, ..., for k = 0, 1, ..."""
    if k < len(_P2_ORDINALS):
        return _P2_ORDINALS[k]
    return "number %d" % (k + 1)


# print_move: one kept move of the loop in words

def print_move(row, position, change, loss_before, loss_after):
    """Print one move of the what-if loop in words, for example
    Ida's first number: 1 less, the loss goes from 25 to 16
    It prints nothing while the change is still ..., a gap in the 🎯 cell of task D."""
    try:
        if change is Ellipsis or row not in (0, 1) or position not in (0, 1):
            return
        name = _p2_name(row, position)
        if change == 1:
            word = "1 more"
        elif change == -1:
            word = "1 less"
        elif change == 0:
            print("%s: stays, the loss stays at %s" % (name, _p2_text(loss_before)))
            return
        else:
            word = "a change of %s" % _p2_short(change)
        print("%s: %s, the loss goes from %s to %s" % (name, word, _p2_text(loss_before), _p2_text(loss_after)))
    except Exception:
        pass


# The rule, kept here for the checks

def _p2_inner(row_a, row_b):
    total = 0
    for i in range(len(row_a)):
        total = total + row_a[i] * row_b[i]
    return total


def _p2_loss(guesses, rating):
    """The loss of a guess for the corner: the rating minus the prediction, squared."""
    difference = rating - _p2_inner(guesses[0], guesses[1])
    return difference * difference


def _p2_move(guesses, rating, row, position):
    """The rule on one number, in place, as what_if_move should do it. Gives back 1, -1 or 0."""
    loss_before = _p2_loss(guesses, rating)
    original = guesses[row][position]
    guesses[row][position] = original + 1
    if _p2_loss(guesses, rating) < loss_before:
        return 1
    guesses[row][position] = original - 1
    if _p2_loss(guesses, rating) < loss_before:
        return -1
    guesses[row][position] = original
    return 0


# The checks

# Each test: the inputs, the expected answer, the call as you would type it, and why that is the answer.
_P2_TESTS = {
    "squared_loss": [
        ((2, -3), 25,
         'squared_loss(2, -3)',
         'The rating 2 and the prediction -3 of the first guess differ by 5, and 5 times 5 is 25'),
        ((2, 2), 0,
         'squared_loss(2, 2)',
         'A prediction that matches the rating misses by 0, and its loss is 0'),
        ((2, 0), 4,
         'squared_loss(2, 0)',
         'A miss of 2 costs 2 times 2, which is 4'),
        ((2, 4), 4,
         'squared_loss(2, 4)',
         'A prediction 2 too high: the difference is -2, and -2 times -2 is 4'),
        ((-1, 2), 9,
         'squared_loss(-1, 2)',
         'The rating -1 and the prediction 2 differ by -3, and -3 times -3 is 9'),
        ((1, 1), 0,
         'squared_loss(1, 1)',
         'The rating 1 and the prediction 1: no miss, no loss'),
        ((2, 1.5), 0.25,
         'squared_loss(2, 1.5)',
         'A prediction with decimals: the difference is 0.5, and 0.5 times 0.5 is 0.25'),
    ],
    "corner_loss": [
        (([[-1, -2], [-1, 2]], 2), 25,
         'corner_loss([[-1, -2], [-1, 2]], 2)',
         'The first guess: the prediction -3, the rating 2'),
        (([[0, -2], [-1, 2]], 2), 36,
         'corner_loss([[0, -2], [-1, 2]], 2)',
         "Ida's first number 1 more: the prediction -4"),
        (([[-2, -2], [-1, 2]], 2), 16,
         'corner_loss([[-2, -2], [-1, 2]], 2)',
         "Ida's first number 1 less: the prediction -2"),
        (([[-2, -1], [-2, 2]], 2), 0,
         'corner_loss([[-2, -1], [-2, 2]], 2)',
         'The end of the three what-ifs: the prediction 2 matches the rating'),
        (([[0, 0], [0, 0]], 2), 4,
         'corner_loss([[0, 0], [0, 0]], 2)',
         'All four at 0: the prediction 0'),
        (([[-3, -3], [-3, 2]], 2), 1,
         'corner_loss([[-3, -3], [-3, 2]], 2)',
         'The prediction 3 misses the rating 2 by 1'),
        (([[-1, -2], [-1, 2]], -2), 1,
         'corner_loss([[-1, -2], [-1, 2]], -2)',
         'The first guess against the rating -2: the prediction -3 misses it by 1'),
        (([[1, 2], [3, 4]], 1), 100,
         'corner_loss([[1, 2], [3, 4]], 1)',
         'Plain numbers: the prediction 1 times 3 plus 2 times 4, 11, misses the rating 1 by 10'),
        (([[0.5, 1], [2, 0.5]], 2), 0.25,
         'corner_loss([[0.5, 1], [2, 0.5]], 2)',
         'Decimals: the prediction 0.5 times 2 plus 1 times 0.5, 1.5, misses the rating 2 by 0.5'),
    ],
    "what_if_move": [
        (([[-1, -2], [-1, 2]], 2, 0, 0), (-1, [[-2, -2], [-1, 2]]),
         'what_if_move([[-1, -2], [-1, 2]], 2, 0, 0)',
         "Ida's first number: 1 more makes the loss 36, more than 25; 1 less makes it 16, less than 25, so the rule keeps 1 less"),
        (([[-2, -2], [-1, 2]], 2, 0, 1), (1, [[-2, -1], [-1, 2]]),
         'what_if_move([[-2, -2], [-1, 2]], 2, 0, 1)',
         "Ida's second number: 1 more makes the loss 4, less than 16, so the rule keeps it"),
        (([[-2, -1], [-1, 2]], 2, 1, 0), (-1, [[-2, -1], [-2, 2]]),
         'what_if_move([[-2, -1], [-1, 2]], 2, 1, 0)',
         "Paddington's first number: 1 more makes the loss 16, more than 4; 1 less makes it 0, less than 4, so the rule keeps 1 less"),
        (([[-3, -3], [-3, 2]], 2, 0, 0), (0, [[-3, -3], [-3, 2]]),
         'what_if_move([[-3, -3], [-3, 2]], 2, 0, 0)',
         "Ida's first number: 1 more makes the loss 4, more than 1, and 1 less makes it 16, more than 1. Neither lowers it, so the number stays"),
        (([[-3, -3], [-3, 2]], 2, 0, 1), (0, [[-3, -3], [-3, 2]]),
         'what_if_move([[-3, -3], [-3, 2]], 2, 0, 1)',
         "Ida's second number: 1 more makes the loss 9, more than 1, and 1 less makes it 1, the same as before. Neither lowers it, so the number stays"),
        (([[1, 1], [0, 1]], 2, 0, 0), (0, [[1, 1], [0, 1]]),
         'what_if_move([[1, 1], [0, 1]], 2, 0, 0)',
         "Ida's first number: 1 more makes the loss 1, the same as before, and 1 less makes it 1, the same as before. Neither lowers it, so the number stays"),
        (([[1, 2], [0, 0]], 2, 1, 1), (1, [[1, 2], [0, 1]]),
         'what_if_move([[1, 2], [0, 0]], 2, 1, 1)',
         "Paddington's second number: 1 more makes the loss 0, less than 4, so the rule keeps it"),
        (([[1, 1], [1, 1]], -1, 0, 0), (-1, [[0, 1], [1, 1]]),
         'what_if_move([[1, 1], [1, 1]], -1, 0, 0)',
         "Ida's first number: 1 more makes the loss 16, more than 9; 1 less makes it 4, less than 9, so the rule keeps 1 less"),
        (([[-2, -1], [-2, 2]], 2, 1, 1), (0, [[-2, -1], [-2, 2]]),
         'what_if_move([[-2, -1], [-2, 2]], 2, 1, 1)',
         "The loss is already 0, and no move can make it smaller, so Paddington's second number stays"),
    ],
    "fit_corner": [
        (([[-1, -2], [-1, 2]], 2), (0, [[-2, -1], [-2, 2]]),
         'fit_corner([[-1, -2], [-1, 2]], 2)',
         'From the first guess the rule keeps three what-ifs, the loss going 25, 16, 4, 0, and ends at [[-2, -1], [-2, 2]]'),
        (([[0, 0], [0, 0]], 2), (4, [[0, 0], [0, 0]]),
         'fit_corner([[0, 0], [0, 0]], 2)',
         'All four at 0: no what-if changes the prediction, so no number moves, and the loop stops after one round, at the loss 4'),
        (([[-3, -3], [-3, 2]], 2), (1, [[-3, -3], [-3, 2]]),
         'fit_corner([[-3, -3], [-3, 2]], 2)',
         'Every hop of 1 overshoots here, so no number moves, and the loop stops after one round, at the loss 1'),
        (([[1, 1], [1, 1]], 2), (0, [[1, 1], [1, 1]]),
         'fit_corner([[1, 1], [1, 1]], 2)',
         'This guess already gives the rating 2: no number needs to move, and the loss is 0'),
        (([[10, 10], [10, 10]], 2), (0, [[1, 1], [1, 1]]),
         'fit_corner([[10, 10], [10, 10]], 2)',
         'From 10, 10, 10 and 10 the rule needs 9 rounds and ends at [[1, 1], [1, 1]]'),
        (([[-1, -2], [-1, 2]], -1), (0, [[-3, -2], [-1, 2]]),
         'fit_corner([[-1, -2], [-1, 2]], -1)',
         'The first guess against the rating -1: two kept what-ifs, and the loss 0 at [[-3, -2], [-1, 2]]'),
        (([[-2, -3], [-2, 3]], 2), (1, [[-3, -2], [-3, 3]]),
         'fit_corner([[-2, -3], [-2, 3]], 2)',
         'Three numbers move in the first round; in the second none can, so the loop stops there, at the loss 1'),
    ],
}

_P2_SUCCESS = {'squared_loss': 'The first guess misses the rating 2 by 5, and its loss is 25.', 'corner_loss': 'The first guess, [[-1, -2], [-1, 2]], has the prediction -3 and the loss 25.', 'what_if_move': 'It keeps 1 more, keeps 1 less, or leaves the number as it was, exactly as the rule says.', 'fit_corner': 'It plays the rule in its order, round after round, and stops when the loss is 0 or when a round passes without a move. From the first guess the loss goes 25, 16, 4, 0.'}


class _P2_Stop(Exception):
    """Raised by the recorder of check_fit_corner to stop a call of fit_corner that cannot go on."""


def _p2_call(fn, inputs):
    """Run a participant's function on fresh copies of the inputs, with its prints silenced. Gives back (answer,
    error, the copies after the call); error is None when the call went through. Never raises."""
    import contextlib, copy, io
    copies = copy.deepcopy(inputs)
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            answer = fn(*copies)
        return answer, None, copies
    except _P2_Stop as stop:
        return None, "STOP " + str(stop), copies
    except Exception as error:
        return None, "%s: %s" % (type(error).__name__, error), copies


def _p2_equal(answer, expected):
    """True when answer is the expected number (decimals within a tiny margin) or the expected list."""
    try:
        if _p2_is_number(expected):
            return _p2_is_number(answer) and abs(answer - expected) < 1e-9
        return bool(answer == expected)
    except Exception:
        return False


def _p2_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    # A gap such as "x = ..." stays in the compiled function; a gap that stands on its own line is only
    # visible in the source, which Jupyter and Colab keep for every cell
    code = getattr(fn, "__code__", None)
    if code is not None and Ellipsis in code.co_consts:
        return True
    import inspect
    try:
        lines = inspect.getsource(fn).splitlines()
    except Exception:
        return False
    for line in lines:
        if "..." in line.split("#")[0]:
            return True
    return False


def _p2_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _p2_grid(value):
    """True when value is two rows of two numbers, the shape of guesses."""
    if not isinstance(value, list) or len(value) != 2:
        return False
    for row in value:
        if not isinstance(row, list) or len(row) != 2:
            return False
        for x in row:
            if not _p2_is_number(x):
                return False
    return True


def _p2_error_hint(error, name):
    """A hint for an error that stopped a participant's function."""
    if "NameError" in error:
        return ("Python does not know a name that %s uses. Run the cells of the tasks above it and the setup "
                "cell of part 2, then the cell of %s, then this check." % (name, name))
    if "unsupported operand" in error and "^" in error:
        return "in Python ^ means something else than a power. Multiply the difference by itself: difference * difference."
    if name == "squared_loss":
        return ("squared_loss gets two numbers: difference = rating - prediction, then loss = difference * "
                "difference, then return loss.")
    if name == "corner_loss":
        if "IndexError" in error:
            return "guesses holds two rows, guesses[0] for Ida and guesses[1] for Paddington, each of two numbers."
        return ("inner_product takes two rows, guesses[0] and guesses[1], and squared_loss takes two numbers, the "
                "rating and the prediction.")
    if name == "what_if_move":
        if "IndexError" in error:
            return ("guesses[row][position] is the number to play with: row 0 or 1 picks the row, position 0 or 1 "
                    "the number in it.")
        return ("corner_loss(guesses, rating) is the loss of the whole guess, one number, and guesses[row][position] "
                "is the one number that changes.")
    return "Each what-if is change = what_if_move(guesses, rating, row, position), inside the two loops."


def _p2_slip(fn, name, slips):
    """The hint of the first slip whose answers fn gives on every whole-number test of name, or None."""
    for slip, hint in slips:
        same = True
        tested = 0
        for inputs, expected, call, why in _P2_TESTS[name]:
            if not all(_p2_is_number(x) and x == int(x) for x in _p2_flat(inputs)):
                continue
            answer, error, after = _p2_call(fn, inputs)
            try:
                wanted = slip(*inputs)
            except Exception:
                same = False
                break
            tested += 1
            if error is not None or not _p2_equal(answer, wanted):
                same = False
                break
        if same and tested > 0:
            return hint
    return None


def _p2_flat(value):
    """The numbers inside a value of nested lists and tuples."""
    if isinstance(value, (list, tuple)):
        out = []
        for x in value:
            out.extend(_p2_flat(x))
        return out
    return [value]


# Task A, squared_loss

_P2_LOSS_SLIPS = [
    (lambda r, p: r - p, "this is the difference, the rating minus the prediction. The loss is the difference "
                         "times itself: loss = difference * difference."),
    (lambda r, p: p - r, "this is the prediction minus the rating, and it is not squared yet. Take the rating "
                         "minus the prediction, then multiply that difference by itself."),
    (lambda r, p: 2 * (r - p), "times 2 doubles the difference. The loss multiplies the difference by itself: "
                               "difference * difference."),
    (lambda r, p: abs(r - p), "this is the size of the miss. The loss is the difference times itself: "
                              "difference * difference."),
    (lambda r, p: (r - p) ^ 2, "in Python ^ means something else than a power. Multiply the difference by "
                               "itself: difference * difference."),
    (lambda r, p: (r + p) * (r + p), "the difference is the rating minus the prediction: rating - prediction."),
    (lambda r, p: p * p, "this is the prediction times itself. Square the difference: difference = rating - "
                         "prediction, then difference * difference."),
    (lambda r, p: r * r - p * p, "square the difference itself: first difference = rating - prediction, then "
                                 "loss = difference * difference."),
]


def _p2_run_squared_loss(fn):
    """None when squared_loss passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P2_TESTS["squared_loss"]:
        answer, error, after = _p2_call(fn, inputs)
        if error is None and _p2_equal(answer, expected):
            continue
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p2_error_hint(error, "squared_loss"))
        if answer is None:
            return ("%s gave no answer (None). Does the function end with return loss?" % call,
                    "the last line is return loss.")
        if not _p2_is_number(answer):
            return ("%s gave %s, expected %s. %s." % (call, _p2_short(answer), _p2_text(expected), why),
                    "return one number, the loss: the difference times itself.")
        hint = _p2_slip(fn, "squared_loss", _P2_LOSS_SLIPS)
        if hint is None:
            hint = "difference = rating - prediction, then loss = difference * difference, then return loss."
        return ("%s gave %s, expected %s. %s." % (call, _p2_text(answer), _p2_text(expected), why), hint)
    return None


# Task B, corner_loss

def _p2_loss_of(r, p):
    return (r - p) * (r - p)


_P2_CORNER_SLIPS = [
    (lambda g, r: _p2_inner(g[0], g[1]),
     "this is the prediction. The loss is one step further: loss = squared_loss(rating, prediction)."),
    (lambda g, r: r - _p2_inner(g[0], g[1]),
     "this is the difference. squared_loss(rating, prediction) squares it for you."),
    (lambda g, r: _p2_loss_of(r, _p2_inner(g[0], g[0])),
     "the prediction pairs Ida's row, guesses[0], with Paddington's row, guesses[1]: inner_product(guesses[0], "
     "guesses[1]). Your code paired Ida's row with itself."),
    (lambda g, r: _p2_loss_of(r, _p2_inner(g[1], g[1])),
     "the prediction pairs Ida's row, guesses[0], with Paddington's row, guesses[1]: inner_product(guesses[0], "
     "guesses[1]). Your code paired Paddington's row with itself."),
    (lambda g, r: _p2_loss_of(_P2_RATING, _p2_inner(g[0], g[1])),
     "use rating, the number that the function gets, in place of %d. The check also tries other ratings."
     % _P2_RATING),
    (lambda g, r: _p2_loss_of(r, g[0][0] * g[1][0]),
     "the prediction adds up both products, the first numbers times each other and the second numbers times "
     "each other: inner_product(guesses[0], guesses[1]) does it."),
    (lambda g, r: _p2_loss_of(r, -3),
     "the prediction must come from the guesses that the function gets: prediction = inner_product(guesses[0], "
     "guesses[1]). Your code used the prediction of the first guess, -3, every time."),
]


def _p2_run_corner_loss(fn):
    """None when corner_loss passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P2_TESTS["corner_loss"]:
        answer, error, after = _p2_call(fn, inputs)
        if error is None and after != inputs:
            return ("%s changed the guesses it got, to %s" % (call, _p2_short(after[0])),
                    "corner_loss only measures a guess: compute the prediction and the loss, and leave guesses as "
                    "they are.")
        if error is None and _p2_equal(answer, expected):
            continue
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p2_error_hint(error, "corner_loss"))
        if answer is None:
            return ("%s gave no answer (None). Does the function end with return loss?" % call,
                    "the last line is return loss.")
        if not _p2_is_number(answer):
            return ("%s gave %s, expected %s. %s." % (call, _p2_short(answer), _p2_text(expected), why),
                    "return one number, the loss: squared_loss(rating, prediction).")
        hint = _p2_slip(fn, "corner_loss", _P2_CORNER_SLIPS)
        if hint is None:
            hint = ("first prediction = inner_product(guesses[0], guesses[1]), then loss = squared_loss(rating, "
                    "prediction), then return loss.")
        return ("%s gave %s, expected %s. %s." % (call, _p2_text(answer), _p2_text(expected), why), hint)
    return None


# Task C, what_if_move

def _p2_three_losses(guesses, rating, row, position):
    """The loss now, with 1 more and with 1 less on guesses[row][position]."""
    import copy
    plus = copy.deepcopy(guesses)
    plus[row][position] = plus[row][position] + 1
    minus = copy.deepcopy(guesses)
    minus[row][position] = minus[row][position] - 1
    return _p2_loss(guesses, rating), _p2_loss(plus, rating), _p2_loss(minus, rating)


def _p2_rule_with(rating):
    """The answers of the rule on every test of what_if_move, with the rating replaced by the given one."""
    import copy
    out = []
    for inputs, expected, call, why in _P2_TESTS["what_if_move"]:
        guesses, r, row, position = copy.deepcopy(inputs)
        change = _p2_move(guesses, rating, row, position)
        out.append((change, guesses))
    return out


def _p2_explain_move(fn, inputs, expected, call, why, answer, error, after):
    """(what went wrong, hint) for the first test of what_if_move that fn fails."""
    guesses, rating, row, position = inputs
    expected_change, expected_after = expected
    wanted = "expected %d, with the guesses %s. %s." % (expected_change, expected_after, why)
    if error is not None:
        return ("%s stopped with an error: %s" % (call, error), _p2_error_hint(error, "what_if_move"))
    if not _p2_grid(after):
        return ("%s left the guesses in another shape, %s" % (call, _p2_short(after)),
                "change one number only, with guesses[row][position] = original + 1 (or original - 1, or "
                "original): guesses must stay two rows of two numbers.")
    got = "%s returned %s and left the guesses at %s; %s" % (call, _p2_short(answer), after, wanted)
    if answer is None:
        return (got, "every case ends with a return line: return 1 in case 1, return -1 in case 2, return 0 in "
                     "case 3.")
    if not _p2_is_number(answer) or answer not in (1, -1, 0):
        return (got, "return the change you kept, one of three numbers: 1 after 1 more, -1 after 1 less, 0 when "
                     "the number stays. Your function returned something else.")
    # The rating typed as a number in the code, in place of the rating that the function gets
    results = []
    for test in _P2_TESTS["what_if_move"]:
        a, e, copies = _p2_call(fn, test[0])
        results.append((a, copies[0]))
    if results == _p2_rule_with(_P2_RATING) and rating != _P2_RATING:
        return (got, "use rating, the number that the function gets, in the two losses: corner_loss(guesses, "
                     "rating). Your function measured the loss against %d every time." % _P2_RATING)
    original = guesses[row][position]
    changed = []
    for i in range(2):
        for j in range(2):
            if after[i][j] != guesses[i][j]:
                changed.append((i, j))
    others = []
    for place in changed:
        if place != (row, position):
            others.append(place)
    if others:
        if others == [(position, row)]:
            return (got, "the number to play with is guesses[row][position], with row first: guesses[%d][%d] here, "
                         "%s. Your function changed guesses[%d][%d]." % (row, position, _p2_name(row, position),
                                                                         position, row))
        return (got, "only the number guesses[row][position] may change: guesses[%d][%d] here, %s."
                % (row, position, _p2_name(row, position)))
    made = after[row][position] - original
    before, plus, minus = _p2_three_losses(guesses, rating, row, position)
    if made != answer:
        if answer == 0 and made == -1:
            return (got, "your function returned 0, but the number is still 1 less than the original. In case "
                         "3, put the original back: guesses[row][position] = original.")
        if answer == 0 and made == 1:
            return (got, "your function returned 0, but the number is still 1 more than the original. Case 2 "
                         "sets it from the original, guesses[row][position] = original - 1, and case 3 puts the "
                         "original back.")
        if answer == -1 and made == 0:
            return (got, "your function returned -1, but the number is back at the original, %s. In case 2, "
                         "subtract 1 from the original: guesses[row][position] = original - 1. After case 1 "
                         "the number holds original + 1, so guesses[row][position] - 1 only undoes the 1 more."
                         % _p2_text(original))
        if made in (1, -1, 0):
            return (got, "the change you return must be the change you made: the number went from %s to %s, a "
                         "change of %s, and your function returned %s." % (_p2_text(original),
                                                                            _p2_text(after[row][position]),
                                                                            _p2_text(made), _p2_text(answer)))
        return (got, "a what-if moves the number by 1 at most, from the original: original + 1, original - 1, or "
                     "the original itself. Here it went from %s to %s." % (_p2_text(original),
                                                                         _p2_text(after[row][position])))
    # The change returned is the change made, and it is the wrong one
    if expected_change == 0:
        if answer == 1:
            mine = plus
        else:
            mine = minus
        if mine == before:
            return (got, "with that move the loss stays at %s. The rule keeps a move only when the loss goes "
                         "down: compare with < (smaller than). With <= an equal loss counts as better too."
                         % _p2_text(before))
        return (got, "with that move the loss grows from %s to %s. The rule keeps a move only when the loss "
                     "goes down: corner_loss(guesses, rating) < loss_before." % (_p2_text(before), _p2_text(mine)))
    if expected_change == -1 and answer == 0:
        return (got, "1 less lowers the loss from %s to %s here, so the rule keeps it and returns -1. In case 2, "
                     "set guesses[row][position] = original - 1, then compare corner_loss(guesses, rating) < "
                     "loss_before. After case 1 the number holds original + 1, so subtracting 1 from "
                     "guesses[row][position] only brings back the original." % (_p2_text(before), _p2_text(minus)))
    if expected_change == 1 and answer == 0:
        return (got, "1 more lowers the loss from %s to %s here, so the rule keeps it and returns 1. In case 1, "
                     "set guesses[row][position] = original + 1, then return 1 when corner_loss(guesses, rating) "
                     "< loss_before." % (_p2_text(before), _p2_text(plus)))
    if expected_change == 1 and answer == -1:
        return (got, "1 more already lowers the loss, from %s to %s, and the rule tries 1 more first: keep it and "
                     "return 1. Is the comparison corner_loss(guesses, rating) < loss_before, smaller than?"
                     % (_p2_text(before), _p2_text(plus)))
    if expected_change == -1 and answer == 1:
        return (got, "1 more makes the loss bigger, from %s to %s, so the rule goes on to 1 less. Keep a move "
                     "only when corner_loss(guesses, rating) < loss_before." % (_p2_text(before), _p2_text(plus)))
    return (got, "remember loss_before and original, then try original + 1, then original - 1, and keep the first "
                 "that makes the loss smaller than loss_before.")


def _p2_run_what_if_move(fn):
    """None when what_if_move passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P2_TESTS["what_if_move"]:
        answer, error, copies = _p2_call(fn, inputs)
        after = copies[0]
        if error is None and _p2_is_number(answer) and answer == expected[0] and after == expected[1]:
            continue
        return _p2_explain_move(fn, inputs, expected, call, why, answer, error, after)
    return None


# Task D, fit_corner

def _p2_call_recorded(fn, inputs, tries):
    """Run fit_corner with what_if_move replaced by a recorder, which plays the rule on one number and notes the
    call: (row, position, change, loss before, loss after). Gives back what _p2_call gives back."""
    space = getattr(fn, "__globals__", None)
    if not isinstance(space, dict):
        return _p2_call(fn, inputs)
    had = "what_if_move" in space
    saved = space.get("what_if_move")

    def recorder(guesses, rating, row, position):
        if len(tries) >= _P2_MAX_TRIES:
            raise _P2_Stop("many")
        if row not in (0, 1) or position not in (0, 1) or isinstance(row, bool) or isinstance(position, bool):
            raise _P2_Stop("call %s %s" % (_p2_short(row), _p2_short(position)))
        before = _p2_loss(guesses, rating)
        change = _p2_move(guesses, rating, row, position)
        tries.append((row, position, change, before, _p2_loss(guesses, rating)))
        return change

    space["what_if_move"] = recorder
    try:
        return _p2_call(fn, inputs)
    finally:
        if had:
            space["what_if_move"] = saved
        else:
            del space["what_if_move"]


def _p2_trace_problem(tries):
    """None when the what-ifs that fit_corner made follow the rule (its order and its two stops), else (what
    went wrong, hint). tries holds (row, position, change, loss before, loss after) for each what-if."""
    order_hint = ("the rule's order is Ida's first number, Ida's second, Paddington's first, Paddington's second, one "
                  "what-if each. The given loops do that, for row outside and for position inside, with the call "
                  "what_if_move(guesses, rating, row, position): row before position.")
    k = 0
    r = 0
    while k < len(tries):
        this_round = tries[k:k + 4]
        for j in range(len(this_round)):
            row, position = this_round[j][0], this_round[j][1]
            if (row, position) != _P2_ORDER[j]:
                want_row, want_position = _P2_ORDER[j]
                return ("in the %s round, the %s what-if was on %s, and the rule's order takes %s there"
                        % (_p2_ordinal(r), _p2_ordinal(j), _p2_name(row, position),
                           _p2_name(want_row, want_position)), order_hint)
        last = this_round[-1]
        if len(this_round) < 4:
            if last[4] != 0:
                return ("it stopped in the middle of the %s round, after the what-if on %s, with the loss at %s"
                        % (_p2_ordinal(r), _p2_name(last[0], last[1]), _p2_text(last[4])),
                        "the two stop lines belong at the end of the round, after both inner loops: indent them as "
                        "far as moved = False.")
            return None
        moved = False
        for t in this_round:
            if t[2] != 0:
                moved = True
        k = k + 4
        if k < len(tries):
            if last[4] == 0:
                return ("the loss was 0 at the end of the %s round, and it went on with another round"
                        % _p2_ordinal(r),
                        "at the end of each round, after the two inner loops, stop with return 0 when "
                        "corner_loss(guesses, rating) == 0.")
            if not moved:
                return ("no number moved in the %s round, so the rule stops there; your fit_corner went on and "
                        "made %d what-ifs in all" % (_p2_ordinal(r), len(tries)),
                        "moved = True belongs inside if change != 0, so that moved stays False in a round without "
                        "a move. At the end of the round, return corner_loss(guesses, rating) when moved == False.")
        elif last[4] != 0 and moved:
            return ("it stopped after the %s round with the loss at %s, although numbers moved in that round: "
                    "the rule goes on with another round" % (_p2_ordinal(r), _p2_text(last[4])),
                    "stop only when the loss is 0 or when no number moved in the round. Is moved = True right "
                    "under if change != 0? Are the stop lines after the two inner loops, with return in them?")
        r = r + 1
    return None


def _p2_run_fit_corner(fn):
    """None when fit_corner passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P2_TESTS["fit_corner"]:
        start, rating = inputs
        expected_loss, expected_after = expected
        tries = []
        answer, error, copies = _p2_call_recorded(fn, inputs, tries)
        after = copies[0]
        if error is not None:
            if error == "STOP many":
                return ("%s made more than %d what-ifs and was stopped. %s." % (call, _P2_MAX_TRIES, why),
                        "the loop must stop: at the end of each round, return 0 when the loss is 0, and return the "
                        "loss when no number moved.")
            if error.startswith("STOP call"):
                return ("%s called what_if_move with a row or a position other than 0 and 1" % call,
                        "pass the four arguments in this order: what_if_move(guesses, rating, row, position).")
            return ("%s stopped with an error: %s" % (call, error), _p2_error_hint(error, "fit_corner"))
        problem = _p2_trace_problem(tries)
        if problem is not None:
            return ("%s: %s. %s." % (call, problem[0], why), problem[1])
        if not _p2_grid(after) or after != expected_after:
            if not tries and _p2_loss(start, rating) != 0:
                return ("%s left the guesses at %s; expected %s. %s." % (call, _p2_short(after), expected_after,
                                                                         why),
                        "inside the two loops, change = what_if_move(guesses, rating, row, position) plays one "
                        "what-if on one number and keeps the move in guesses.")
            return ("%s left the guesses at %s; expected %s. %s." % (call, _p2_short(after), expected_after, why),
                    "fit_corner changes guesses itself, through what_if_move(guesses, rating, row, position): work "
                    "on guesses, the list that the function gets.")
        if answer is None:
            return ("%s gave no answer (None). %s." % (call, why),
                    "return the loss at the end: return 0 when the loss is 0, return corner_loss(guesses, rating) "
                    "when no number moved. A bare return gives back None.")
        if not _p2_equal(answer, expected_loss):
            return ("%s returned %s, and the loss at the end is %s. %s." % (call, _p2_short(answer),
                                                                           _p2_text(expected_loss), why),
                    "return the loss at the end: return 0 when the loss is 0, return corner_loss(guesses, rating) "
                    "when no number moved.")
    return None


_P2_RUNNERS = {"squared_loss": _p2_run_squared_loss, "corner_loss": _p2_run_corner_loss,
               "what_if_move": _p2_run_what_if_move, "fit_corner": _p2_run_fit_corner}
_P2_NEEDS = {"squared_loss": [],
             "corner_loss": [("inner_product", "part 1"), ("squared_loss", "task A")],
             "what_if_move": [("corner_loss", "task B")],
             "fit_corner": [("what_if_move", "task C"), ("corner_loss", "task B")]}


def _p2_works(name):
    """True when the function called name exists and passes its tests."""
    fn = globals().get(name)
    if not callable(fn):
        return False
    if name == "inner_product":
        for row_a, row_b in (([2, 1], [1, 2]), ([-1, -2], [-1, 2]), ([3], [-2]), ([0.5, 1], [2, 0.5])):
            answer, error, after = _p2_call(fn, (row_a, row_b))
            if error is not None or not _p2_equal(answer, _p2_inner(row_a, row_b)):
                return False
        return True
    if _p2_has_gap(fn):
        return False
    return _P2_RUNNERS[name](fn) is None


def _p2_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    if _p2_has_gap(fn):
        print("⏳ %s is not finished yet: replace each ... (Python calls it Ellipsis) with your code, run its "
              "cell, then run this check again." % name)
        return
    for other, place in _P2_NEEDS[name]:
        if not _p2_uses(fn, other) or _p2_works(other):
            continue
        if other == "inner_product":
            print("⏳ %s uses inner_product, which does not work any more. Run its cell in part 1 again (it is "
                  "given, so undo any change to it), then run this check again." % name)
        else:
            print("⏳ %s uses your %s from %s, which does not work yet. Get the check of %s to ✅ first, then run "
                  "this check again." % (name, other, place, place))
        return
    failure = _P2_RUNNERS[name](fn)
    if failure is None:
        print("✅ %s works on %d tests. %s" % (name, len(_P2_TESTS[name]), _P2_SUCCESS[name]))
        return
    print("❌ " + failure[0])
    print("   💡 Hint: " + failure[1])


def check_squared_loss(fn):
    """Task A: tests squared_loss on seven pairs of a rating and a prediction."""
    try:
        _p2_check("squared_loss", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 2 again, then this cell." % error)


def check_corner_loss(fn):
    """Task B: tests corner_loss on nine guesses for the corner, some with other ratings."""
    try:
        _p2_check("corner_loss", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 2 again, then this cell." % error)


def check_what_if_move(fn):
    """Task C: tests what_if_move on nine numbers: 1 more kept, 1 less kept, and numbers that must stay."""
    try:
        _p2_check("what_if_move", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 2 again, then this cell." % error)


def check_fit_corner(fn):
    """Task D: tests fit_corner on seven guesses, from the first guess to a far one, and the two stops."""
    try:
        _p2_check("fit_corner", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 2 again, then this cell." % error)


if _p2_missing:
    print("Note: part 1 has not run in this session, so this cell defined %s for part 2. To run part 1 as well, "
          "click on this cell, choose Runtime > Run before in the menu, then run this cell again."
          % " and ".join(_p2_missing))
print("✅ Setup of part 2 complete.")

## The ratings

Three customers, Ida, Raj and Elsa, rated two films, *Paddington* (2014) and *The Martian* (2015), with the five marks of the game turned into
numbers: two ticks 2, a tick 1, a shrug 0, a cross -1, two crosses -2. Nobody filled in a profile, and no critic
described the films. `ratings` has one row per customer and one column per film. The customers and their
ratings are invented for the lesson.

In [ ]:
# The ratings of section 2 of the game (invented for the lesson): one row per customer, one column per film
customer_names2 = ["Ida", "Raj", "Elsa"]
film_names2 = ["Paddington", "The Martian"]
ratings = [
    [ 2,  1],   # Ida
    [-1,  2],   # Raj
    [ 1, -2],   # Elsa
]

In [ ]:
show_matrix(ratings, customer_names2, film_names2)

## One corner

As in the game, we start with one corner of the table: Ida and *Paddington*. Ida is row 0 and *Paddington* column 0, so her
rating is `ratings[0][0]`.

In [ ]:
# The corner: Ida (row 0) and Paddington (column 0)
rating = ratings[0][0]
print("Ida gave Paddington", rating)

## The first guess

The Matchmaker guesses two numbers for Ida and two for *Paddington*. In Python the guess is `guesses`, a list of two
rows, stacked as in the game: `guesses[0]` holds Ida's two numbers, on top, and `guesses[1]` holds Paddington's two
numbers, below. So `guesses[0][1]` is Ida's second number. The cell below holds the Matchmaker's first guess from
the game.

In [ ]:
# The Matchmaker's first guess for the corner: Ida's two numbers on top, Paddington's two below
guesses = [[-1, -2], [-1, 2]]
corner_names = ["Ida", "Paddington"]
number_names = ["first number", "second number"]
show_matrix(guesses, corner_names, number_names)

In [ ]:
# The prediction: the inner product of Ida's row and Paddington's row
prediction = inner_product(guesses[0], guesses[1])
print("The prediction:", prediction)
print("The rating:    ", rating)

**Reading the result.** -3. Ida's row `[-1, -2]` sits on top of Paddington's row `[-1, 2]`: the products are 1 and -4, and their sum is
-3. Ida gave *Paddington* 2, so the guess misses: the difference, the rating minus the prediction, is 2 - (-3) = 5.

## 🎯 Task A. The squared loss

How wrong is a prediction? The **difference** is the rating minus the prediction, 5 for the first guess. The
**loss** is the difference times itself, 5 times 5, which is 25: the squared loss of the game's diagram.

Write `squared_loss(rating, prediction)`. It gets two numbers, the rating and the prediction, and returns the
loss.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# The difference is the rating minus the prediction: for the first guess, 2 - (-3) = 5.
# In Python: difference = rating - prediction
# The loss is that difference times itself.

In [ ]:
#@title 💡 Hint 2 for task A: the shape of the code (double-click to open) { display-mode: "form" }
#     difference = rating - prediction
#     loss = ...                         # the difference times itself, with *
#     return loss

In [ ]:
# 🎯 YOUR TURN, task A: the squared loss
def squared_loss(rating, prediction):
    # rating and prediction are two numbers. Return the difference times itself.
    difference = ...   # 🎯 YOUR TURN: the rating minus the prediction
    loss = ...         # 🎯 YOUR TURN: the difference times itself
    return loss

In [ ]:
check_squared_loss(squared_loss)

In [ ]:
# The loss of the first guess, and of three other predictions
print("Prediction -3, a miss of 5: the loss is", squared_loss(rating, prediction))
print("Prediction 1, a miss of 1: the loss is", squared_loss(rating, 1))
print("Prediction 0, a miss of 2: the loss is", squared_loss(rating, 0))
print("Prediction 4, 2 too high:  the loss is", squared_loss(rating, 4))

**Reading the result.** 25 for the first guess. A miss of 1 costs 1 and a miss of 2 costs 4: twice the miss,
four times the loss. That is why we square: bigger mistakes are punished harder. A prediction of 4, 2 too high,
costs 4 as well, because -2 times -2 is 4: too high and too low count alike.

## 🎯 Task B. The loss of a guess

The loss of a whole guess for the corner takes two steps: the prediction, which is the inner product of the two
rows, then the squared loss of the rating and that prediction. Write `corner_loss(guesses, rating)`. For the
first guess, `corner_loss(guesses, rating)` gives 25.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# Two steps, as in the cells above. First the prediction: the inner product of guesses[0],
# Ida's row, and guesses[1], Paddington's row. Then the loss: squared_loss of the rating and that prediction.

In [ ]:
#@title 💡 Hint 2 for task B: the shape of the code (double-click to open) { display-mode: "form" }
#     prediction = inner_product(guesses[0], guesses[1])
#     loss = squared_loss(..., ...)
#     return loss

In [ ]:
# 🎯 YOUR TURN, task B: the loss of a guess for the corner
def corner_loss(guesses, rating):
    # guesses[0] is Ida's row of two numbers, guesses[1] is Paddington's row.
    prediction = ...   # 🎯 YOUR TURN: the inner product of the two rows (use inner_product)
    loss = ...         # 🎯 YOUR TURN: the squared loss of the rating and the prediction
    return loss

In [ ]:
check_corner_loss(corner_loss)

## What if?

💭 **Think first.** What happens to the prediction if we add 1 to Ida's first number? Will the loss go up or
down? In the stack, Ida's first number sits on top of Paddington's first number, -1. Then run the cell below. It tries
the what-if by hand on a copy of the first guess, `trial`: 1 more, then 1 less than the original number.

In [ ]:
# The what-if by hand: Ida's first number, 1 more, then 1 less than the original
trial = [[-1, -2], [-1, 2]]                # the first guess again
original = trial[0][0]                      # Ida's first number, -1
print("As guessed:", trial, "  the loss is", corner_loss(trial, rating))
trial[0][0] = original + 1
print("1 more:    ", trial, "  the loss is", corner_loss(trial, rating))
trial[0][0] = original - 1
print("1 less:    ", trial, "  the loss is", corner_loss(trial, rating))

**Reading the result.** 1 more makes the prediction -4. In the stack only one product changes: Ida's first
number meets Paddington's first number, -1, so one more of it adds -1 to the prediction. Adding 1 to a number always
adds its partner to the prediction. Here that moves the prediction away from the rating 2, and the loss grows
from 25 to 36: worse. 1 less does the opposite: the prediction rises to -2, and the loss falls to 16: better.
That is the what-if of the game, and the rule keeps the 1 less.

## 🎯 Task C. One what-if

Now the rule of the game, for any one of the four numbers. The number is `guesses[row][position]`: `row` 0 is
Ida's row and 1 Paddington's, `position` 0 is the first number of the row and 1 the second. So Ida's first number is
`guesses[0][0]`, and Paddington's second number is `guesses[1][1]`.

**The rule.** Add 1; if the loss goes down, keep it. Otherwise subtract 1 from the original; if the loss goes
down, keep that. Otherwise leave the number as it was.

Write `what_if_move(guesses, rating, row, position)`. It changes the number in `guesses` itself, so the move it
keeps stays in the list, and it returns the change it kept: `1`, `-1` or `0`. The skeleton names the three
cases; fill in their lines the way the cell above did by hand. A move counts only when the loss goes **down**,
so compare with `<`, smaller than.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# Before any change, remember two things: the loss, loss_before = corner_loss(guesses, rating),
# and the number itself, original = guesses[row][position] (that line is given).
# Then each case sets the number from original: original + 1, original - 1, or original.

In [ ]:
#@title 💡 Hint 2 for task C (double-click to open) { display-mode: "form" }
# Case 1 in Python:
#     guesses[row][position] = original + 1
#     if corner_loss(guesses, rating) < loss_before:
#         return 1
# Case 2 is the same with original - 1 and return -1.
# Case 3 puts the original back: guesses[row][position] = original

In [ ]:
#@title 💡 Hint 3 for task C: the shape of the code (double-click to open) { display-mode: "form" }
#     loss_before = corner_loss(guesses, rating)
#     original = guesses[row][position]
#     guesses[row][position] = original + 1
#     if corner_loss(guesses, rating) < loss_before:
#         return 1
#     guesses[row][position] = original - 1
#     if ...:                                    # the loss is now smaller than loss_before
#         return -1
#     guesses[row][position] = original
#     return 0

In [ ]:
# 🎯 YOUR TURN, task C: one what-if on one guessed number
def what_if_move(guesses, rating, row, position):
    # The number to play with is guesses[row][position]. Change it as the rule says,
    # and return the change you kept: 1, -1 or 0.
    loss_before = ...                    # 🎯 YOUR TURN: the loss of the guesses now (use corner_loss)
    original = guesses[row][position]    # the number as it is now

    # Case 1: 1 more. If the loss goes down, keep it.
    ...                                  # 🎯 YOUR TURN: make the number 1 more than the original
    # 🎯 YOUR TURN: if the loss is now smaller than loss_before, return 1
    ...

    # Case 2: 1 less than the original. If the loss goes down, keep it.
    ...                                  # 🎯 YOUR TURN: make the number 1 less than the original
    # 🎯 YOUR TURN: if the loss is now smaller than loss_before, return -1
    ...

    # Case 3: neither helped. Leave the number as it was.
    ...                                  # 🎯 YOUR TURN: put the original back
    return 0

In [ ]:
check_what_if_move(what_if_move)

In [ ]:
# The game's three worked what-ifs on the first guess, one call each
guesses = [[-1, -2], [-1, 2]]
change = what_if_move(guesses, rating, 0, 0)
print("Ida's first number:        change", change, "  the loss is now", corner_loss(guesses, rating))
change = what_if_move(guesses, rating, 0, 1)
print("Ida's second number:       change", change, "  the loss is now", corner_loss(guesses, rating))
change = what_if_move(guesses, rating, 1, 0)
print("Paddington's first number: change", change, "  the loss is now", corner_loss(guesses, rating))
show_matrix(guesses, corner_names, number_names)

**Reading the result.** Three what-ifs, as in the game: Ida's first number 1 less (the loss 25 to 16), Ida's second number 1 more (the loss 16 to 4) and Paddington's first number 1 less (the loss 4 to 0). Ida's row is now `[-2, -1]` and Paddington's row `[-2, 2]`: the
products 4 and -2 add up to 2, the rating. The guess was corrected three times, and now the prediction matches.

## 🎯 Task D. Again and again

The rule visits the four numbers in a fixed order, Ida's first number, her second number, Paddington's first number and its second number, and then starts again: one pass over the
four is a **round**. It stops when the loss is 0, or after a round in which no number moved, because the next
round would then do exactly the same.

Write `fit_corner(guesses, rating)`. The skeleton gives the rounds, `for round_number in range(100)`: at most 100
rounds, far more than this notebook needs, since from any guess of numbers from -2 to 2 the rule needs at most
3 rounds, and from 10, 10, 10 and 10 it needs 9. Inside a round, two loops visit the four numbers in the
rule's order, and `moved` remembers whether a number moved. A given line, `print_move`, prints every move that
is kept. You fill in the what-if, `moved`, and the two stops. `!=` means "is not equal to", and `return` ends
the function at once, also in the middle of a loop: that is how the loop stops. `fit_corner` changes `guesses`
itself and returns the loss at the end.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task D (double-click to open) { display-mode: "form" }
# Inside the two loops, change = what_if_move(guesses, rating, row, position) plays one what-if
# on guesses[row][position] and gives back the change it kept.
# When that change is not 0, a number moved: moved = True.

In [ ]:
#@title 💡 Hint 2 for task D (double-click to open) { display-mode: "form" }
# The two stops come at the end of a round, after the two inner loops, as far indented as moved = False:
#         if corner_loss(guesses, rating) == 0:
#             return 0
# and a second if, which returns corner_loss(guesses, rating) when moved == False.

In [ ]:
#@title 💡 Hint 3 for task D: the shape of the code (double-click to open) { display-mode: "form" }
#                 change = what_if_move(guesses, rating, row, position)
#                 if change != 0:
#                     moved = True
#                     ...                    # the two given lines that print the move
#         if corner_loss(guesses, rating) == 0:
#             return 0
#         if moved == False:
#             return ...                     # the loss of the guesses

In [ ]:
# 🎯 YOUR TURN, task D: the what-if loop on the corner
def fit_corner(guesses, rating):
    # Play what-if on the four numbers in the rule's order, round after round. Stop when the
    # loss is 0, or after a round in which no number moved. Return the loss at the end.
    for round_number in range(100):                 # at most 100 rounds
        moved = False                               # no number has moved in this round yet
        for row in range(2):                        # row 0 is Ida's, row 1 Paddington's
            for position in range(2):               # position 0 is the first number, 1 the second
                loss_before = corner_loss(guesses, rating)
                change = ...                        # 🎯 YOUR TURN: one what-if on guesses[row][position]
                if change != 0:
                    ...                             # 🎯 YOUR TURN: remember that a number moved
                    loss_after = corner_loss(guesses, rating)
                    print_move(row, position, change, loss_before, loss_after)
        # 🎯 YOUR TURN: if the loss is now 0, return 0
        ...
        # 🎯 YOUR TURN: if no number moved in this round, return the loss
        ...
    return corner_loss(guesses, rating)

In [ ]:
check_fit_corner(fit_corner)

In [ ]:
# The what-if loop on the first guess
guesses = [[-1, -2], [-1, 2]]
final_loss = fit_corner(guesses, rating)
print("The loss at the end:", final_loss)
show_matrix(guesses, corner_names, number_names)

**Reading the result.** 25, 16, 4, 0: the three what-ifs of the game, found by the loop on its own, all in the first
round. The loop then stops at the end of that round, because the loss is 0: the prediction matches the rating.

## 🎯 Your own guess

In the game you typed a guess of your own for Ida and *Paddington*. Type its four numbers into the cell below, in place
of the Matchmaker's first guess, and run the cell. If you skipped that step of the game, pick any four numbers
from -2 to 2. Run the cell again whenever you change the numbers.

### 🎯 Your turn

In [ ]:
# 🎯 YOUR TURN: your own guess from the game, in place of the Matchmaker's first guess
my_guesses = [[-1, -2],     # 🎯 YOUR TURN: Ida's first and second number
              [-1, 2]]      # 🎯 YOUR TURN: Paddington's first and second number

print("Your guess: the loss is", corner_loss(my_guesses, rating))
final_loss = fit_corner(my_guesses, rating)
print("The loss at the end:", final_loss)
show_matrix(my_guesses, corner_names, number_names)

**Reading the result.** Every move that the loop keeps lowers the loss, so the losses printed above go down,
move after move. From every guess of numbers from -2 to 2 they reach 0 within 3 rounds, except from one: all
four at 0, the first experiment below. Your loop may well end at other numbers than the first guess's: Ida's
row `[-2, -1]` with Paddington's `[-2, 2]` gives 2, and so do many other pairs of rows. The rating says only what the inner
product must be.

## Two experiments: when the loop cannot move

💭 **Think first.** In both guesses below the loop stops at once. Before you run the cells, guess why.

1. All four numbers at 0, the loss 4. What does 1 more or 1 less on any number do to the prediction?
2. Ida -3 and -3 with Paddington -3 and 2: the prediction is 3, a miss of only 1, and the loss is 1. Why can a hop of 1
   not fix such a small miss?

In [ ]:
# Experiment 1: all four numbers at 0
zeros = [[0, 0], [0, 0]]
print("The loss at the end:", fit_corner(zeros, rating))
show_matrix(zeros, corner_names, number_names)

**Reading the result.** At 0 the loop cannot move. Every number's partner is 0, so 1 more or 1 less on any
number adds 0 to the prediction: it stays 0, and the loss stays 4. The rule leaves every number as it was, and
after one round without a move the loop stops.

In [ ]:
# Experiment 2: a guess that misses the rating by only 1
stuck = [[-3, -3], [-3, 2]]
print("The prediction:", inner_product(stuck[0], stuck[1]))
print("The loss at the end:", fit_corner(stuck, rating))
show_matrix(stuck, corner_names, number_names)

**Reading the result.** The loop stops at once, at the loss 1. A hop of 1 on a number adds its partner to the
prediction or takes it away, and the partners here are 2 and 3 in size: the prediction jumps from 3 to 0, 1, 5 or 6, over the
rating 2 or away from it. No hop lowers the loss, so no number moves. Hops of one whole unit are too clumsy
here.

## Back to the game

That is part 2: your `fit_corner` guesses, measures the error, corrects the guess and goes again, Popper's
schema at work, until the prediction matches the rating. With hops of one whole unit it can get stuck. Section
3 of the game (optional) makes the hops smaller, and section 4 does every customer and every film at once. Part
3 (optional) and part 4 of this notebook come after them.

# Part 3. Smaller hops: gradient descent (optional)

In part 2 the what-if loop got stuck on a guess that missed the rating by only 1: every hop of one whole unit
jumped over the bottom of the valley. In section 3 of the game a grasshopper hops a tiny step, **eps**, feels how
steep the ground is, the **slope**, and then jumps downhill by a small amount, **alpha** times the slope. Done
again and again, that is **gradient descent**. (eps is short for epsilon, the Greek letter ε.)

This part is optional: part 4 does not need it. If you do it, you teach the computer the grasshopper's moves on
the corner of part 2, Ida and *Paddington*:

- **Task A**: the slope from a tiny hop;
- **Task B**: the slope from a formula;
- **Task C**: one step downhill for all four guessed numbers at once;
- **Task D**: gradient descent, step after step, until the loss is about 0.

Then five experiments. Part 3 stands on its own: it gives again what it needs from parts 1 and 2. Run its setup
cell first.

In [ ]:
#@title Setup of part 3 (run this first) { display-mode: "form" }
# The plumbing of part 3: print_hop, print_slopes and print_step, which print results in words, show_losses,
# which prints the list of losses of gradient descent, and the checks of tasks A to D. You do not need to read
# it. A check prints one short message and never stops the notebook.

# Part 3 uses two things from part 1: the given inner_product and show_matrix. When part 1 has not run in this
# session, this cell defines them, so that part 3 works on its own. Part 2's loss comes again in the next cell.
_p3_missing = []
if "inner_product" not in globals():
    _p3_missing.append("inner_product")

    def inner_product(row_a, row_b):
        # row_a and row_b are two lists of numbers of the same length.
        # Multiply them position by position, then add up the products.
        total = 0
        for i in range(len(row_a)):
            total = total + row_a[i] * row_b[i]
        return total

if "show_matrix" not in globals() or "plain stand-in" in str(getattr(show_matrix, "__doc__", "")):
    if "show_matrix" not in globals():
        _p3_missing.append("show_matrix")

    def show_matrix(matrix, row_names=None, column_names=None):
        """A plain stand-in for the show_matrix of part 1: the table as text, decimals rounded to 2 places."""
        try:
            fine = isinstance(matrix, list) and len(matrix) > 0
            if fine:
                for row in matrix:
                    if not isinstance(row, list) or len(row) == 0 or len(row) != len(matrix[0]):
                        fine = False
                    else:
                        for x in row:
                            if x is Ellipsis or x is None or isinstance(x, (list, tuple)):
                                fine = False
            if not fine:
                print("⏳ Nothing to show yet: %s. Finish the 🎯 cell above, run it, then run this cell again."
                      % _p3_short(matrix))
                return
            rows, columns = len(matrix), len(matrix[0])
            if not isinstance(row_names, (list, tuple)) or len(row_names) != rows:
                row_names = [str(i) for i in range(rows)]
            if not isinstance(column_names, (list, tuple)) or len(column_names) != columns:
                column_names = [str(j) for j in range(columns)]
            entries = []
            for i in range(rows):
                line = []
                for j in range(columns):
                    if isinstance(matrix[i][j], float):
                        line.append("%.2f" % matrix[i][j])
                    else:
                        line.append(str(matrix[i][j]))
                entries.append(line)
            width = max(len(str(name)) for name in row_names)
            widths = []
            for j in range(columns):
                w = len(str(column_names[j]))
                for i in range(rows):
                    w = max(w, len(entries[i][j]))
                widths.append(w)
            print(" " * width + "".join("   " + str(column_names[j]).rjust(widths[j]) for j in range(columns)))
            for i in range(rows):
                print(str(row_names[i]).ljust(width)
                      + "".join("   " + entries[i][j].rjust(widths[j]) for j in range(columns)))
        except Exception as error:
            print("The table could not be shown (%s)." % error)

_P3_ROW_NAMES = ['Ida', 'Paddington']
_P3_POSITION_NAMES = ["first", "second"]
_P3_RATING = 2
_P3_HOPS = [0.01, 0.1, 0.001]           # the game's hop, then the two other hops the check of task A tries
_P3_TOL = 0.0001                 # below this the loss counts as 0
_P3_MARGIN = 1e-9                 # two decimal numbers count as equal within this relative margin
_P3_MAX_CALLS = 5000              # the check of task D stops a loop that makes more steps
_P3_ORDER = [(0, 0), (0, 1), (1, 0), (1, 1)]


def _p3_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = repr(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 80:
        text = text[:77] + "..."
    return text


def _p3_is_number(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)


def _p3_dec(x, places):
    """x rounded to at most places decimals, without trailing zeros: -5.91, 0.0591, 4."""
    if not _p3_is_number(x):
        return _p3_short(x)
    if x != x or x == float("inf") or x == float("-inf"):
        return str(x)
    text = "%.*f" % (places, x)
    if "." in text:
        text = text.rstrip("0").rstrip(".")
    if text == "-0":
        text = "0"
    return text


def _p3_loss_text(x):
    """A loss as text: at most 4 decimals, two digits for a small one, and about 0 below 0.0001."""
    if _p3_is_number(x) and 0 <= x < _P3_TOL:
        return "about 0"
    if _p3_is_number(x) and 0 < x < 0.001:
        return _p3_dec(x, 5)
    return _p3_dec(x, 4)


def _p3_name(row, position):
    """Ida's first number"""
    return "%s's %s number" % (_P3_ROW_NAMES[row], _P3_POSITION_NAMES[position])


_P3_NAME_WIDTH = max(len(_p3_name(r, p)) for r, p in _P3_ORDER) + 1


def _p3_count(n, word):
    """1 step, 4 steps"""
    if n == 1:
        return "1 " + word
    return "%d %ss" % (n, word)


def _p3_grid(value):
    """True when value is two rows of two numbers, the shape of guesses."""
    if not isinstance(value, list) or len(value) != 2:
        return False
    for row in value:
        if not isinstance(row, list) or len(row) != 2:
            return False
        for x in row:
            if not _p3_is_number(x):
                return False
    return True


def _p3_waiting(value):
    """Why a value is not a number yet, in words."""
    if value is Ellipsis:
        return "it is still ..., a gap in a 🎯 cell"
    if value is None:
        return "it is None (does the function end with a return line?)"
    return "it is %s" % _p3_short(value)


# The four helpers that print results in words

def print_hop(eps, slope_found):
    """One line of the grasshopper's table: the hop and the slope it measured, at most 4 decimals."""
    try:
        if not _p3_is_number(slope_found):
            print("A hop of %s: ⏳ no slope yet, %s. Finish task A, run its cell, then run this cell again."
                  % (_p3_dec(eps, 6), _p3_waiting(slope_found)))
            return
        print("A hop of %s gives the slope %s" % (_p3_dec(eps, 6).ljust(6), _p3_dec(slope_found, 4)))
    except Exception:
        pass


def print_slopes(row, position, from_formula, from_hop):
    """One line of the table of slopes: the number, its slope from the formula and its slope from a tiny hop."""
    try:
        name = (_p3_name(row, position) + ":").ljust(_P3_NAME_WIDTH)
        waiting = []
        if not _p3_is_number(from_formula):
            waiting.append("slope, task B")
        if not _p3_is_number(from_hop):
            waiting.append("slope_by_hop, task A")
        if waiting:
            print("%s ⏳ not finished yet: %s" % (name, " and ".join(waiting)))
            return
        print("%s the slope %s by a tiny hop %s" % (name, _p3_dec(from_formula, 4).ljust(6), _p3_dec(from_hop, 4)))
    except Exception:
        pass


def print_step(step_number, guesses, loss):
    """One step of gradient descent in words, for example
    Step  1:  the loss 0.0553   the prediction 1.7648   Ida -2.88 and -3.08, Paddington -2.88 and 2.12
    It prints nothing while the loss is still ..., a gap in the 🎯 cell of task D, and one calm line at step 1
    when the loss is something else."""
    try:
        if not _p3_is_number(loss):
            if loss is not Ellipsis and step_number == 1:
                print("Step  1:  ⏳ no loss yet: %s. The checks of tasks C and D say more." % _p3_waiting(loss))
            return
        text = "Step %2s:  the loss %s" % (step_number, _p3_loss_text(loss).ljust(8))
        if _p3_grid(guesses):
            text = text + "  the prediction %s  %s %s and %s, %s %s and %s" % (
                _p3_dec(_p3_inner(guesses[0], guesses[1]), 4).ljust(8),
                _P3_ROW_NAMES[0], _p3_dec(guesses[0][0], 2), _p3_dec(guesses[0][1], 2),
                _P3_ROW_NAMES[1], _p3_dec(guesses[1][0], 2), _p3_dec(guesses[1][1], 2))
        print(text.rstrip())
    except Exception:
        pass


def show_losses(losses):
    """The list of losses that gradient_descent gives back, in one line: the number of steps, then the losses,
    the loss at the start first (all of them, or the first five and the last three)."""
    try:
        if losses is Ellipsis or losses is None or not isinstance(losses, list) or len(losses) == 0:
            if isinstance(losses, list):
                reason = "the list of losses is empty"
            else:
                reason = "gradient_descent gave no list yet: " + _p3_waiting(losses)
            print("⏳ Nothing to show yet: %s. Finish task D, run its cell, then run this cell again." % reason)
            return
        for x in losses:
            if not _p3_is_number(x):
                print("⏳ The list of losses still holds %s. The check of task D says more." % _p3_short(x))
                return
        texts = []
        for x in losses:
            texts.append(_p3_loss_text(x))
        if len(texts) > 9:
            texts = texts[:5] + ["..."] + texts[-3:]
        steps = len(losses) - 1
        if steps == 0:
            print("No step. The loss: %s" % texts[0])
        else:
            print("%s. The losses: %s" % (_p3_count(steps, "step"), ", ".join(texts)))
    except Exception as error:
        print("The losses could not be shown (%s)." % error)


# The arithmetic, kept here for the checks

def _p3_inner(row_a, row_b):
    total = 0
    for i in range(len(row_a)):
        total = total + row_a[i] * row_b[i]
    return total


def _p3_loss(guesses, rating):
    """The loss of a guess for the corner: the rating minus the prediction, squared."""
    difference = rating - _p3_inner(guesses[0], guesses[1])
    return difference * difference


def _p3_with(guesses, row, position, amount):
    """A copy of guesses with amount added to guesses[row][position]."""
    out = [list(guesses[0]), list(guesses[1])]
    out[row][position] = out[row][position] + amount
    return out


def _p3_slope(guesses, rating, row, position):
    difference = rating - _p3_inner(guesses[0], guesses[1])
    return -2 * guesses[1 - row][position] * difference


def _p3_step(guesses, rating, alpha):
    """One step of gradient descent, in place, as gradient_step should do it. Gives back the loss after it."""
    slopes = [[0, 0], [0, 0]]
    for row, position in _P3_ORDER:
        slopes[row][position] = _p3_slope(guesses, rating, row, position)
    for row, position in _P3_ORDER:
        guesses[row][position] = guesses[row][position] - alpha * slopes[row][position]
    return _p3_loss(guesses, rating)


def _p3_step_variant(guesses, rating, alpha, how):
    """The guesses after one step done with a slip: how is "one at a time", "plus", "no alpha" or "divided"."""
    g = [list(guesses[0]), list(guesses[1])]
    if how == "one at a time":
        for row, position in _P3_ORDER:
            g[row][position] = g[row][position] - alpha * _p3_slope(g, rating, row, position)
        return g
    slopes = [[0, 0], [0, 0]]
    for row, position in _P3_ORDER:
        slopes[row][position] = _p3_slope(g, rating, row, position)
    for row, position in _P3_ORDER:
        if how == "plus":
            g[row][position] = g[row][position] + alpha * slopes[row][position]
        elif how == "no alpha":
            g[row][position] = g[row][position] - slopes[row][position]
        elif how == "divided":
            g[row][position] = g[row][position] - slopes[row][position] / alpha
    return g


# The checks

# Each test: the inputs, the expected answer, the call as you would type it, and why that is the answer.
_P3_TESTS = {
    "slope_by_hop": [
        (([[-3, -3], [-3, 2]], 2, 0, 0, 0.01), -5.9099999999998705,
         'slope_by_hop([[-3, -3], [-3, 2]], 2, 0, 0, 0.01)',
         "The game's hop: the loss goes from 1 to 0.9409, a change of -0.0591, and -0.0591 / 0.01 is -5.91"),
        (([[-3, -3], [-3, 2]], 2, 0, 0, 0.1), -5.100000000000009,
         'slope_by_hop([[-3, -3], [-3, 2]], 2, 0, 0, 0.1)',
         'Another hop, 0.1: the loss goes from 1 to 0.49, and -0.51 / 0.1 is -5.1'),
        (([[-3, -3], [-3, 2]], 2, 0, 0, 0.001), -5.991000000000191,
         'slope_by_hop([[-3, -3], [-3, 2]], 2, 0, 0, 0.001)',
         'Another hop, 0.001: -0.005991 / 0.001 is -5.991'),
        (([[-3, -3], [-3, 2]], 2, 1, 0, 0.01), -5.9099999999998705,
         'slope_by_hop([[-3, -3], [-3, 2]], 2, 1, 0, 0.01)',
         "Paddington's first number: its partner is Ida's first, -3, so it gets the same slope as Ida's first number"),
        (([[-3, -3], [-3, 2]], 2, 0, 1, 0.01), 4.03999999999991,
         'slope_by_hop([[-3, -3], [-3, 2]], 2, 0, 1, 0.01)',
         "Ida's second number: its partner is Paddington's second, 2, and the slope by this hop is 4.04"),
        (([[-1, -2], [-1, 2]], 2, 1, 1, 0.01), 20.03999999999948,
         'slope_by_hop([[-1, -2], [-1, 2]], 2, 1, 1, 0.01)',
         "Part 2's first guess, Paddington's second number: the slope by this hop is 20.04"),
        (([[1, 1], [1, 1]], -1, 0, 0, 0.01), 6.009999999999849,
         'slope_by_hop([[1, 1], [1, 1]], -1, 0, 0, 0.01)',
         'Another rating, -1: the slope by this hop is 6.01'),
        (([[-2.88, -3.08], [-2.88, 2.12]], 2, 1, 1, 0.01), 1.5436959999999667,
         'slope_by_hop([[-2.88, -3.08], [-2.88, 2.12]], 2, 1, 1, 0.01)',
         'A guess with decimals, one step downhill from the stuck guess: the slope by this hop is 1.5437'),
    ],
    "slope": [
        (([[-3, -3], [-3, 2]], 2, 0, 0), -6,
         'slope([[-3, -3], [-3, 2]], 2, 0, 0)',
         "The stuck guess, Ida's first number: -2 times its partner (-3) times the difference (-1) is -6"),
        (([[-3, -3], [-3, 2]], 2, 0, 1), 4,
         'slope([[-3, -3], [-3, 2]], 2, 0, 1)',
         "The stuck guess, Ida's second number: -2 times its partner (2) times the difference (-1) is 4"),
        (([[-3, -3], [-3, 2]], 2, 1, 0), -6,
         'slope([[-3, -3], [-3, 2]], 2, 1, 0)',
         "The stuck guess, Paddington's first number: -2 times its partner (-3) times the difference (-1) is -6"),
        (([[-3, -3], [-3, 2]], 2, 1, 1), -6,
         'slope([[-3, -3], [-3, 2]], 2, 1, 1)',
         "The stuck guess, Paddington's second number: -2 times its partner (-3) times the difference (-1) is -6"),
        (([[-1, -2], [-1, 2]], 2, 0, 0), 10,
         'slope([[-1, -2], [-1, 2]], 2, 0, 0)',
         "Part 2's first guess, Ida's first number: -2 times its partner (-1) times the difference (5) is 10"),
        (([[-1, -2], [-1, 2]], 2, 0, 1), -20,
         'slope([[-1, -2], [-1, 2]], 2, 0, 1)',
         "Part 2's first guess, Ida's second number: -2 times its partner (2) times the difference (5) is -20"),
        (([[-1, -2], [-1, 2]], 2, 1, 0), 10,
         'slope([[-1, -2], [-1, 2]], 2, 1, 0)',
         "Part 2's first guess, Paddington's first number: -2 times its partner (-1) times the difference (5) is 10"),
        (([[-1, -2], [-1, 2]], 2, 1, 1), 20,
         'slope([[-1, -2], [-1, 2]], 2, 1, 1)',
         "Part 2's first guess, Paddington's second number: -2 times its partner (-2) times the difference (5) is 20"),
        (([[1, 1], [1, 1]], -1, 1, 1), 6,
         'slope([[1, 1], [1, 1]], -1, 1, 1)',
         "Another rating, -1, Paddington's second number: -2 times its partner (1) times the difference (-3) is 6"),
        (([[-2.88, -3.08], [-2.88, 2.12]], 2, 0, 1), -0.9972480000000032,
         'slope([[-2.88, -3.08], [-2.88, 2.12]], 2, 0, 1)',
         "A guess with decimals, Ida's second number: -2 times its partner (2.12) times the difference (0.2352) is -0.997248"),
        (([[-2, -1], [-2, 2]], 2, 0, 0), 0,
         'slope([[-2, -1], [-2, 2]], 2, 0, 0)',
         'This guess gives the rating exactly: the difference is 0, so every slope is 0, the bottom of the valley'),
        (([[0, 0], [0, 0]], 2, 1, 0), 0,
         'slope([[0, 0], [0, 0]], 2, 1, 0)',
         'All four at 0: the partner is 0, so the slope is 0'),
    ],
    "gradient_step": [
        (([[-3, -3], [-3, 2]], 2, 0.02), (0.05531904000000035, [[-2.88, -3.08], [-2.88, 2.12]]),
         'gradient_step([[-3, -3], [-3, 2]], 2, 0.02)',
         'One step from the stuck guess: the slopes -6, 4, -6 and -6, so Ida -2.88 and -3.08, Paddington -2.88 and 2.12, and the loss 0.0553'),
        (([[-1, -2], [-1, 2]], 2, 0.02), (9.734400000000003, [[-1.2, -1.6], [-1.2, 1.6]]),
         'gradient_step([[-1, -2], [-1, 2]], 2, 0.02)',
         "Part 2's first guess: the slopes 10, -20, 10 and 20, so Ida -1.2 and -1.6, Paddington -1.2 and 1.6, and the loss 9.7344"),
        (([[-3, -3], [-3, 2]], 2, 0.1), (25.8064, [[-2.4, -3.4], [-2.4, 2.6]]),
         'gradient_step([[-3, -3], [-3, 2]], 2, 0.1)',
         'A big step, alpha 0.1: Ida -2.4 and -3.4, Paddington -2.4 and 2.6, and the loss jumps to 25.8064'),
        (([[0, 0], [0, 0]], 2, 0.02), (4.0, [[0.0, 0.0], [0.0, 0.0]]),
         'gradient_step([[0, 0], [0, 0]], 2, 0.02)',
         'All four at 0: every slope is 0, so no number moves, and the loss stays 4'),
        (([[1, 1], [1, 1]], -1, 0.05), (3.9204, [[0.7, 0.7], [0.7, 0.7]]),
         'gradient_step([[1, 1], [1, 1]], -1, 0.05)',
         'Another rating, -1, and another step size, 0.05: every slope is 6, so every number becomes 0.7'),
        (([[-2.88, -3.08], [-2.88, 2.12]], 2, 0.02), (0.0027620280320285163, [[-2.90709504, -3.06005504], [-2.90709504, 2.09102336]]),
         'gradient_step([[-2.88, -3.08], [-2.88, 2.12]], 2, 0.02)',
         'The second step from the stuck guess: the loss falls from 0.0553 to 0.0028'),
    ],
    "gradient_descent": [
        (([[-3, -3], [-3, 2]], 2, 0.02, 100), ([1, 0.05531904000000035, 0.0027620280320285163, 0.00014029558408464734, 7.096244225543689e-06], [[-2.9023581895511557, -3.063457044455762], [-2.9023581895511557, 2.0960043146053766]]),
         'gradient_descent([[-3, -3], [-3, 2]], 2, 0.02, 100)',
         'From the stuck guess the loss is below 0.0001 after 4 steps: 1, 0.0553, 0.0028, 0.00014, about 0'),
        (([[-1, -2], [-1, 2]], 2, 0.02, 100), ([25, 9.734400000000003, 4.575509425594344, 2.2282869381221087, 1.0789733931627243, 0.5130212150797621, 0.23914170798084858, 0.10957071728443307, 0.04953024234347733, 0.022167571221124623, 0.009851186384560104, 0.0043563619484046695, 0.0019200021176837222, 0.0008442960439546177, 0.0003707039893276124, 0.0001625993225817433, 7.12716215423046e-05], [[-1.7631362247403484, -1.0569255409688034], [-1.7631362247403484, 1.0569255409688034]]),
         'gradient_descent([[-1, -2], [-1, 2]], 2, 0.02, 100)',
         "From part 2's first guess the loss is below 0.0001 after 16 steps"),
        (([[0, 0], [0, 0]], 2, 0.02, 5), ([4, 4.0, 4.0, 4.0, 4.0, 4.0], [[0.0, 0.0], [0.0, 0.0]]),
         'gradient_descent([[0, 0], [0, 0]], 2, 0.02, 5)',
         'All four at 0: no number moves, the loss stays 4, and the loop makes all 5 steps'),
        (([[-1, -2], [-1, 2]], 2, 0.02, 3), ([25, 9.734400000000003, 4.575509425594344, 2.2282869381221087], [[-1.46524784359637, -1.2805061537274265], [-1.46524784359637, 1.2805061537274265]]),
         'gradient_descent([[-1, -2], [-1, 2]], 2, 0.02, 3)',
         'At most 3 steps: after them the loss, 2.2283, is still above 0.0001, so the list holds 4 losses'),
        (([[-3, -3], [-3, 2]], 2, 0.1, 3), ([1, 25.8064, 486.5594015046958, 77270.00315614797], [[16.5067740020736, 3.0108866789375983], [16.5067740020736, 2.491371321753598]]),
         'gradient_descent([[-3, -3], [-3, 2]], 2, 0.1, 3)',
         'A step that is too big, alpha 0.1: the losses grow, 1, 25.8064, 486.5594, 77270.0032'),
        (([[-1, -2], [-1, 2]], -1, 0.02, 100), ([4, 1.4864486399999997, 0.6091710259532649, 0.26096153149821866, 0.11439243525280497, 0.05079540781725127, 0.02272863107825644, 0.010218013680397781, 0.004607365352144285, 0.0020814814769897837, 0.0009415370345153641, 0.00042624750877792494, 0.00019307459128392227, 8.748784497057738e-05], [[-1.2400480202706419, -1.5959550702660323], [-1.2400480202706419, 1.5959550702660323]]),
         'gradient_descent([[-1, -2], [-1, 2]], -1, 0.02, 100)',
         'Another rating, -1: the loss is below 0.0001 after 13 steps'),
        (([[1, 2], [-1, -2]], 2, 0.02, 10), ([49, 21.086463999999996, 13.892412417573809, 10.566486347751631, 8.68207307309436, 7.488784066914709, 6.6777660740884865, 6.099019311058385, 5.671159068123074, 5.346337729107161, 5.094649343766063], [[0.22677433366338107, 0.45354866732676213], [-0.22677433366338107, -0.45354866732676213]]),
         'gradient_descent([[1, 2], [-1, -2]], 2, 0.02, 10)',
         'A flipped guess: the loss falls towards 4 and never gets below 0.0001, so the loop makes all 10 steps'),
    ],
}

_P3_SUCCESS = {'slope_by_hop': "The game's hop of 0.01 gives -5.91, and the guesses stay as they were.", 'slope': 'At the stuck guess the four slopes are -6, 4, -6 and -6.', 'gradient_step': 'From the stuck guess one step gives Ida -2.88 and -3.08, Paddington -2.88 and 2.12, and the loss 0.0553.', 'gradient_descent': 'From the stuck guess the loss is below 0.0001 after 4 steps.'}


class _P3_Stop(Exception):
    """Raised by the recorder of check_gradient_descent to stop a call that cannot go on."""


def _p3_call(fn, inputs):
    """Run a participant's function on fresh copies of the inputs, with its prints silenced. Gives back (answer,
    error, the copies after the call); error is None when the call went through. Never raises."""
    import contextlib, copy, io
    copies = copy.deepcopy(inputs)
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            answer = fn(*copies)
        return answer, None, copies
    except _P3_Stop as stop:
        return None, "STOP " + str(stop), copies
    except Exception as error:
        return None, "%s: %s" % (type(error).__name__, error), copies


def _p3_close(a, b):
    """True when a and b are numbers that agree within the margin (relative, and absolute near 0)."""
    if not _p3_is_number(a) or not _p3_is_number(b):
        return False
    try:
        return abs(a - b) <= _P3_MARGIN * max(1.0, abs(b))
    except Exception:
        return False


def _p3_close_grid(a, b):
    """True when a and b are both two rows of two numbers that agree within the margin."""
    if not _p3_grid(a) or not _p3_grid(b):
        return False
    for row, position in _P3_ORDER:
        if not _p3_close(a[row][position], b[row][position]):
            return False
    return True


def _p3_close_list(a, b):
    """True when a and b are lists of numbers of the same length that agree within the margin."""
    if not isinstance(a, list) or not isinstance(b, list) or len(a) != len(b):
        return False
    for k in range(len(a)):
        if not _p3_close(a[k], b[k]):
            return False
    return True


def _p3_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    # A gap such as "x = ..." stays in the compiled function; a gap that stands on its own line is only
    # visible in the source, which Jupyter and Colab keep for every cell
    code = getattr(fn, "__code__", None)
    if code is not None and Ellipsis in code.co_consts:
        return True
    import inspect
    try:
        lines = inspect.getsource(fn).splitlines()
    except Exception:
        return False
    for line in lines:
        if "..." in line.split("#")[0]:
            return True
    return False


def _p3_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _p3_grid_text(g):
    """[[-2.88, -3.08], [-2.88, 2.12]], with at most 4 decimals."""
    if not _p3_grid(g):
        return _p3_short(g)
    return "[[%s, %s], [%s, %s]]" % (_p3_dec(g[0][0], 4), _p3_dec(g[0][1], 4), _p3_dec(g[1][0], 4),
                                     _p3_dec(g[1][1], 4))


def _p3_list_text(values):
    """[1, 0.0553, 0.0028, ...], with at most 6 decimals, cut short when it is long."""
    if not isinstance(values, list):
        return _p3_short(values)
    texts = []
    for x in values:
        if _p3_is_number(x):
            texts.append(_p3_dec(x, 6))
        else:
            texts.append(_p3_short(x))
    if len(texts) > 9:
        texts = texts[:5] + ["..."] + texts[-3:]
    return "[" + ", ".join(texts) + "]"


def _p3_error_hint(error, name):
    """A hint for an error that stopped a participant's function."""
    if "UnboundLocalError" in error or "before assignment" in error or "not callable" in error:
        return ("a variable and a function share a name. Inside a function, slope = ... makes slope a number, "
                "and Python forgets the function slope there. Give the variable another name, for example s. "
                "If you typed slope = ... in a cell, run the cell of task B again to bring the function back.")
    if "NameError" in error:
        return ("Python does not know a name that %s uses. Run the setup cell of part 3, the cell that gives "
                "corner_loss, and the cells of the tasks above, then the cell of %s, then this check."
                % (name, name))
    if "ZeroDivisionError" in error:
        return "divide by eps, the size of the hop, and only by eps."
    if "IndexError" in error:
        return ("guesses[row][position] is one number: row 0 or 1 picks the row, position 0 or 1 the number in it. "
                "The partner is guesses[1 - row][position].")
    if name == "slope_by_hop":
        return ("corner_loss(guesses, rating) takes the whole guess, and guesses[row][position] is the one number "
                "that hops.")
    if name == "slope":
        return ("inner_product takes two rows, guesses[0] and guesses[1]; the partner is one number, "
                "guesses[1 - row][position].")
    if name == "gradient_step":
        return ("slope(guesses, rating, row, position) takes the whole guess and the place of one number, and "
                "gives back one number.")
    return ("gradient_step(guesses, rating, alpha) takes the whole guess and gives back one number, the loss after "
            "the step: loss = gradient_step(guesses, rating, alpha), then losses.append(loss).")


def _p3_slip(fn, name, slips):
    """The hint of the first slip whose answers fn gives on every test of name, or None."""
    for slip, hint in slips:
        same = True
        for inputs, expected, call, why in _P3_TESTS[name]:
            answer, error, after = _p3_call(fn, inputs)
            try:
                wanted = slip(*inputs)
            except Exception:
                same = False
                break
            if error is not None or not _p3_close(answer, wanted):
                same = False
                break
        if same:
            return hint
    return None


# Task A, slope_by_hop

def _p3_hop_slips():
    L = _p3_loss
    W = _p3_with
    return [
        (lambda g, r, row, p, eps: L(W(g, row, p, eps), r) - L(g, r),
         "this is the change of the loss. The slope is the change per unit of hop: return change / eps."),
        (lambda g, r, row, p, eps: (L(g, r) - L(W(g, row, p, eps), r)) / eps,
         "the sign is flipped. The change is the loss after the hop minus the loss before: change = loss_after - "
         "loss_before."),
        (lambda g, r, row, p, eps: L(g, r) - L(W(g, row, p, eps), r),
         "the change is the loss after the hop minus the loss before, loss_after - loss_before, and the slope is "
         "that change divided by eps."),
        (lambda g, r, row, p, eps: (L(W(g, row, p, eps), r) - L(g, r)) * eps,
         "the change was multiplied by eps. The slope is the change per unit of hop: divide, change / eps."),
        (lambda g, r, row, p, eps: (L(W(g, row, p, 1), r) - L(g, r)) / eps,
         "the number hopped by 1, part 2's what-if. The grasshopper hops by eps: guesses[row][position] = "
         "original + eps."),
        (lambda g, r, row, p, eps: L(W(g, row, p, 1), r) - L(g, r),
         "the number hopped by 1, part 2's what-if. The grasshopper hops by eps, original + eps, and the slope is "
         "the change divided by eps."),
        (lambda g, r, row, p, eps: (L(W(g, p, row, eps), r) - L(g, r)) / eps,
         "the number that hops is guesses[row][position], with row first. Your code hopped guesses[position][row]."),
        (lambda g, r, row, p, eps: (L(W(g, 0, 0, eps), r) - L(g, r)) / eps,
         "the number that hops is guesses[row][position]: row and position say which of the four numbers it is. "
         "Your code always hopped guesses[0][0], Ida's first number."),
        (lambda g, r, row, p, eps: (L(W(g, row, p, eps), r) - L(g, r)) / _P3_HOPS[0],
         "divide by eps, the hop that the function gets, in place of %s. The check also tries the hops %s and %s."
         % (_P3_HOPS[0], _P3_HOPS[1], _P3_HOPS[2])),
        (lambda g, r, row, p, eps: (L(W(g, row, p, _P3_HOPS[0]), r) - L(g, r)) / eps,
         "hop by eps, the hop that the function gets: original + eps, in place of %s. The check also tries the hops "
         "%s and %s." % (_P3_HOPS[0], _P3_HOPS[1], _P3_HOPS[2])),
        (lambda g, r, row, p, eps: 0,
         "the loss after the hop came out equal to the loss before. Measure it after the line that hops: "
         "loss_after = corner_loss(guesses, rating), right below guesses[row][position] = original + eps."),
        (lambda g, r, row, p, eps: L(W(g, row, p, eps), r),
         "this is the loss after the hop. Return the change per unit of hop: (loss_after - loss_before) / eps."),
    ]


def _p3_run_slope_by_hop(fn):
    """None when slope_by_hop passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P3_TESTS["slope_by_hop"]:
        answer, error, copies = _p3_call(fn, inputs)
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p3_error_hint(error, "slope_by_hop"))
        after = copies[0]
        if _p3_close(answer, expected):
            if not _p3_close_grid(after, inputs[0]):
                return ("%s gave the right slope, %s, but left the guesses at %s; they should stay %s"
                        % (call, _p3_dec(expected, 6), _p3_grid_text(after), _p3_grid_text(inputs[0])),
                        "the grasshopper only feels the ground and hops back: after the loss after the hop is "
                        "measured, put the original back, guesses[row][position] = original.")
            continue
        if answer is None:
            return ("%s gave no answer (None). Does the function end with a return line?" % call,
                    "the last line returns the change divided by eps: return change / eps.")
        if not _p3_is_number(answer):
            return ("%s gave %s, expected %s. %s." % (call, _p3_short(answer), _p3_dec(expected, 6), why),
                    "return one number, the slope: the change of the loss divided by eps.")
        hint = _p3_slip(fn, "slope_by_hop", _p3_hop_slips())
        if hint is None:
            hint = ("loss_before, then the hop original + eps, then loss_after, then the hop back, then "
                    "(loss_after - loss_before) / eps.")
        return ("%s gave %s, expected %s. %s." % (call, _p3_dec(answer, 6), _p3_dec(expected, 6), why), hint)
    return None


# Task B, slope

def _p3_slope_slips():
    P = _p3_inner
    return [
        (lambda g, r, row, p: -2 * g[row][p] * (r - P(g[0], g[1])),
         "you used the number itself, guesses[row][position]. The formula takes its partner, the number it meets "
         "in the stack: the same position in the other row, guesses[1 - row][position]."),
        (lambda g, r, row, p: -2 * g[row][1 - p] * (r - P(g[0], g[1])),
         "you used the other number of the same row. The partner sits in the other row, at the same position: "
         "guesses[1 - row][position]."),
        (lambda g, r, row, p: -2 * g[1 - row][1 - p] * (r - P(g[0], g[1])),
         "the partner sits in the other row at the same position, guesses[1 - row][position]. Your code took the "
         "other position too."),
        (lambda g, r, row, p: g[1 - row][p] * (r - P(g[0], g[1])),
         "the -2 is missing: return -2 * partner * difference."),
        (lambda g, r, row, p: 2 * g[1 - row][p] * (r - P(g[0], g[1])),
         "the sign is flipped. The difference is the rating minus the prediction, rating - prediction, and the "
         "factor is -2, with its minus: return -2 * partner * difference."),
        (lambda g, r, row, p: -1 * g[1 - row][p] * (r - P(g[0], g[1])),
         "your slopes are half as big: the factor is -2. Return -2 * partner * difference."),
        (lambda g, r, row, p: -2 * g[1 - row][p] * P(g[0], g[1]),
         "the formula takes the difference, the rating minus the prediction: difference = rating - prediction."),
        (lambda g, r, row, p: -2 * g[1 - row][p] * (_P3_RATING - P(g[0], g[1])),
         "use rating, the number that the function gets, in place of %d. The check also tries another rating."
         % _P3_RATING),
    ]


def _p3_run_slope(fn):
    """None when slope passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P3_TESTS["slope"]:
        answer, error, copies = _p3_call(fn, inputs)
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p3_error_hint(error, "slope"))
        if not _p3_close_grid(copies[0], inputs[0]):
            return ("%s changed the guesses it got, to %s" % (call, _p3_grid_text(copies[0])),
                    "slope only measures: compute the difference and the partner, and leave guesses as they are.")
        if _p3_close(answer, expected):
            continue
        if answer is None:
            return ("%s gave no answer (None). Does the function end with a return line?" % call,
                    "the last line is return -2 * partner * difference.")
        if not _p3_is_number(answer):
            return ("%s gave %s, expected %s. %s." % (call, _p3_short(answer), _p3_dec(expected, 6), why),
                    "return one number: -2 times the partner times the difference.")
        hint = _p3_slip(fn, "slope", _p3_slope_slips())
        if hint is None and abs(answer - expected) <= 0.01 * max(1.0, abs(expected)):
            hint = ("close: a tiny hop gives nearly the slope. Task B asks for the exact slope, from the formula: "
                    "return -2 * partner * difference.")
        if hint is None:
            hint = ("difference = rating - prediction, partner = guesses[1 - row][position], then return -2 * "
                    "partner * difference.")
        return ("%s gave %s, expected %s. %s." % (call, _p3_dec(answer, 6), _p3_dec(expected, 6), why), hint)
    return None


# Task C, gradient_step

_P3_STEP_SLIPS = [
    ("one at a time",
     "close, but each slope must be measured on the guess as it was before the step. Measure all four slopes "
     "first, in the first loop, and move the numbers in the second loop with the slopes you measured: "
     "guesses[row][position] - alpha * slopes[row][position]."),
    ("plus",
     "the grasshopper went uphill: every number moved with its slope, and the loss grew. Move against the slope, "
     "with a minus: guesses[row][position] - alpha * slopes[row][position]."),
    ("no alpha",
     "every number lost its whole slope, a giant jump. The step is alpha times the slope: "
     "guesses[row][position] - alpha * slopes[row][position]."),
    ("divided",
     "the slope was divided by alpha, a giant jump. The step is alpha times the slope: "
     "guesses[row][position] - alpha * slopes[row][position]."),
]


def _p3_run_gradient_step(fn):
    """None when gradient_step passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P3_TESTS["gradient_step"]:
        start, rating, alpha = inputs
        expected_loss, expected_after = expected
        answer, error, copies = _p3_call(fn, inputs)
        after = copies[0]
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p3_error_hint(error, "gradient_step"))
        wanted = "expected %s. %s." % (_p3_grid_text(expected_after), why)
        if not _p3_grid(after):
            return ("%s left the guesses in another shape, %s; %s" % (call, _p3_short(after), wanted),
                    "move each number with guesses[row][position] = guesses[row][position] - alpha * "
                    "slopes[row][position]: guesses must stay two rows of two numbers.")
        if not _p3_close_grid(after, expected_after):
            got = "%s left the guesses at %s; %s" % (call, _p3_grid_text(after), wanted)
            if _p3_close_grid(after, start):
                return (got, "no number moved. In the second loop, set guesses[row][position] = "
                             "guesses[row][position] - alpha * slopes[row][position].")
            for how, hint in _P3_STEP_SLIPS:
                same = True
                for test in _P3_TESTS["gradient_step"]:
                    a, e, c = _p3_call(fn, test[0])
                    if e is not None or not _p3_close_grid(c[0], _p3_step_variant(test[0][0], test[0][1],
                                                                                  test[0][2], how)):
                        same = False
                        break
                if same:
                    return (got, hint)
            for row in range(2):
                if (_p3_close(after[row][0], start[row][0]) and _p3_close(after[row][1], start[row][1])
                        and not _p3_close_grid(start, expected_after)):
                    other = []
                    for position in range(2):
                        other.append(_p3_close(expected_after[row][position], start[row][position]))
                    if other == [False, False]:
                        return (got, "%s's two numbers did not move. The two loops visit both rows, row 0 and "
                                     "row 1, and every number moves." % _P3_ROW_NAMES[row])
            near = True
            for row, position in _P3_ORDER:
                if abs(after[row][position] - expected_after[row][position]) > 0.05 * max(
                        1.0, abs(expected_after[row][position])):
                    near = False
            if near:
                return (got, "close, but off. Measure every slope with slope(guesses, rating, row, position), the "
                             "formula of task B, all four before any number moves, then move each number by "
                             "alpha times its slope.")
            return (got, "first slopes[row][position] = slope(guesses, rating, row, position) for all four, then "
                         "guesses[row][position] = guesses[row][position] - alpha * slopes[row][position].")
        if answer is None:
            return ("%s moved the numbers right but gave no answer (None)" % call,
                    "end the function with return corner_loss(guesses, rating), the loss after the step.")
        if not _p3_close(answer, expected_loss):
            if _p3_close(answer, _p3_loss(start, rating)):
                return ("%s moved the numbers right but returned %s, the loss before the step; expected %s"
                        % (call, _p3_dec(answer, 6), _p3_dec(expected_loss, 6)),
                        "return the loss after the step: corner_loss(guesses, rating) on the last line, after the "
                        "numbers have moved.")
            return ("%s moved the numbers right but returned %s; expected the loss after the step, %s"
                    % (call, _p3_short(answer), _p3_dec(expected_loss, 6)),
                    "end the function with return corner_loss(guesses, rating), the loss after the step.")
    return None


# Task D, gradient_descent

def _p3_call_recorded(fn, inputs, calls):
    """Run gradient_descent with gradient_step replaced by a recorder, which makes the step right and notes the
    call (the rating and alpha it got). Gives back what _p3_call gives back."""
    space = getattr(fn, "__globals__", None)
    if not isinstance(space, dict):
        return _p3_call(fn, inputs)
    had = "gradient_step" in space
    saved = space.get("gradient_step")

    def recorder(guesses, rating, alpha):
        if len(calls) >= _P3_MAX_CALLS:
            raise _P3_Stop("many")
        if not _p3_grid(guesses) or not _p3_is_number(rating) or not _p3_is_number(alpha):
            raise _P3_Stop("call")
        calls.append((rating, alpha))
        return _p3_step(guesses, rating, alpha)

    space["gradient_step"] = recorder
    try:
        return _p3_call(fn, inputs)
    finally:
        if had:
            space["gradient_step"] = saved
        else:
            del space["gradient_step"]


def _p3_run_gradient_descent(fn):
    """None when gradient_descent passes every test, else (what went wrong, hint)."""
    stop_hint = ("stop when the loss is below 0.0001: at the top of each round of the loop, if loss < 0.0001: "
                 "return losses. And loss must hold the loss after the latest step: loss = "
                 "gradient_step(guesses, rating, alpha).")
    for inputs, expected, call, why in _P3_TESTS["gradient_descent"]:
        start, rating, alpha, max_steps = inputs
        expected_losses, expected_end = expected
        calls = []
        answer, error, copies = _p3_call_recorded(fn, inputs, calls)
        after = copies[0]
        if error is not None:
            if error == "STOP many":
                return ("%s made more than %d steps and was stopped. %s." % (call, _P3_MAX_CALLS, why),
                        "the loop makes at most max_steps steps, for step_number in range(max_steps), and stops "
                        "early when the loss is below 0.0001.")
            if error == "STOP call":
                return ("%s called gradient_step with something other than a guess, a rating and a step size"
                        % call, "pass the three arguments in this order: gradient_step(guesses, rating, alpha).")
            return ("%s stopped with an error: %s" % (call, error), _p3_error_hint(error, "gradient_descent"))
        for used_rating, used_alpha in calls:
            if used_alpha != alpha:
                return ("%s called gradient_step with the step size %s; the function got %s"
                        % (call, _p3_short(used_alpha), _p3_short(alpha)),
                        "use alpha, the step size that gradient_descent gets: gradient_step(guesses, rating, "
                        "alpha). The check also tries other step sizes.")
            if used_rating != rating:
                return ("%s called gradient_step with the rating %s; the function got %s"
                        % (call, _p3_short(used_rating), _p3_short(rating)),
                        "use rating, the number that gradient_descent gets: gradient_step(guesses, rating, alpha). "
                        "The check also tries another rating.")
        if len(calls) > max_steps:
            return ("%s made %d steps, and max_steps was %d. %s." % (call, len(calls), max_steps, why),
                    "the given loop, for step_number in range(max_steps), makes at most max_steps steps: keep it "
                    "as it is.")
        if answer is None:
            return ("%s gave no answer (None). %s." % (call, why),
                    "return the list: return losses, both inside the stop and on the last line.")
        if not isinstance(answer, list):
            if _p3_is_number(answer):
                given = "the number %s" % _p3_dec(answer, 6)
            else:
                given = _p3_short(answer)
            return ("%s gave %s; expected the list of losses, %s. %s." % (call, given,
                                                                        _p3_list_text(expected_losses), why),
                    "return losses, the whole list, the loss at the start first.")
        for x in answer:
            if not _p3_is_number(x):
                return ("%s gave a list that holds %s; expected the losses, %s" % (call, _p3_short(x),
                                                                                  _p3_list_text(expected_losses)),
                        "each entry of losses is one number, a loss: losses.append(loss), with loss = "
                        "gradient_step(guesses, rating, alpha).")
        if _p3_close_list(answer, expected_losses):
            if not _p3_close_grid(after, expected_end):
                return ("%s gave the right losses, but left the guesses at %s; expected %s"
                        % (call, _p3_grid_text(after), _p3_grid_text(expected_end)),
                        "gradient_descent changes guesses itself, through gradient_step(guesses, rating, alpha): "
                        "work on guesses, the list that the function gets.")
            continue
        got = "%s gave %s; expected %s. %s." % (call, _p3_list_text(answer), _p3_list_text(expected_losses), why)
        n, n_expected = len(answer), len(expected_losses)
        if not calls:
            return (got, "no step was made. In the loop, loss = gradient_step(guesses, rating, alpha) makes one step "
                         "and gives back the loss after it.")
        if n == n_expected - 1 and _p3_close_list(answer, expected_losses[1:]):
            return (got, "the loss at the start is missing: the list begins with it, losses = [loss], before the "
                         "loop.")
        if n == 1 and _p3_close(answer[0], expected_losses[0]):
            return (got, "losses holds only the loss at the start. Put each new loss at the end of the list, "
                         "losses.append(loss), right after the step.")
        repeats = True
        for x in answer:
            if not _p3_close(x, answer[0]):
                repeats = False
        if n > 1 and repeats and not _p3_close_list(answer, expected_losses[:n]):
            return (got, "the list repeats the loss at the start: loss = gradient_step(guesses, rating, alpha) keeps "
                         "the loss after each step, and losses.append(loss) puts it in the list.")
        if n > n_expected and _p3_close_list(answer[:n_expected], expected_losses):
            return ("%s went on after the loss was below 0.0001: the list holds %d losses, and %d were expected. "
                    "%s." % (call, n, n_expected, why), stop_hint)
        if n < n_expected and _p3_close_list(answer, expected_losses[:n]):
            return ("%s stopped after %s, with the loss at %s, above 0.0001. %s."
                    % (call, _p3_count(n - 1, "step"), _p3_loss_text(answer[-1]), why),
                    "return early only when the loss is below 0.0001: if loss < 0.0001: return losses. "
                    "Otherwise the loop goes on, and the last line, return losses, comes after max_steps steps.")
        return (got, "losses = [loss] before the loop; in each round, first the stop, then loss = "
                     "gradient_step(guesses, rating, alpha), then losses.append(loss).")
    return None


_P3_RUNNERS = {"slope_by_hop": _p3_run_slope_by_hop, "slope": _p3_run_slope,
               "gradient_step": _p3_run_gradient_step, "gradient_descent": _p3_run_gradient_descent}
_P3_NEEDS = {"slope_by_hop": [("corner_loss", "")],
             "slope": [("inner_product", "")],
             "gradient_step": [("slope", "task B"), ("corner_loss", "")],
             "gradient_descent": [("gradient_step", "task C"), ("corner_loss", "")]}
_P3_TASK = {"slope_by_hop": "A", "slope": "B", "gradient_step": "C", "gradient_descent": "D"}
_P3_GIVEN = {"inner_product": "in part 1 (or in this setup, when part 1 has not run)",
             "corner_loss": "at the top of part 3"}


def _p3_works(name):
    """True when the function called name exists and passes its tests."""
    fn = globals().get(name)
    if not callable(fn):
        return False
    if name in ("inner_product", "corner_loss"):
        if name == "inner_product":
            cases = [(([2, 1], [1, 2]), 4), (([-1, -2], [-1, 2]), -3), (([3], [-2]), -6), (([0.5, 1], [2, 0.5]), 1.5)]
        else:
            cases = []
            for g, r in (([[-3, -3], [-3, 2]], 2), ([[-1, -2], [-1, 2]], 2), ([[0.5, 1], [2, 0.5]], 2),
                         ([[1, 1], [1, 1]], -1)):
                cases.append(((g, r), _p3_loss(g, r)))
        for inputs, wanted in cases:
            answer, error, after = _p3_call(fn, inputs)
            if error is not None or not _p3_close(answer, wanted):
                return False
        return True
    if _p3_has_gap(fn):
        return False
    return _P3_RUNNERS[name](fn) is None


def _p3_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        if _p3_is_number(fn):
            got = "the number %s" % _p3_dec(fn, 6)
        else:
            got = _p3_short(fn)
        print("❌ check_%s needs the function %s itself, and it got %s. Write check_%s(%s), with no brackets "
              "after %s. If a cell set %s = ..., run the cell of task %s again to bring the function back."
              % (name, name, got, name, name, name, name, _P3_TASK[name]))
        return
    if _p3_has_gap(fn):
        print("⏳ %s is not finished yet: replace each ... (Python calls it Ellipsis) with your code, run its "
              "cell, then run this check again." % name)
        return
    for other, place in _P3_NEEDS[name]:
        if not _p3_uses(fn, other) or _p3_works(other):
            continue
        if other in _P3_GIVEN:
            print("⏳ %s uses %s, which is missing or was changed. Run its cell %s again (it is given, so undo any "
                  "change to it), then run this check again." % (name, other, _P3_GIVEN[other]))
        else:
            print("⏳ %s uses your %s from %s, which does not work yet. Get the check of %s to ✅ first, then run "
                  "this check again." % (name, other, place, place))
        return
    failure = _P3_RUNNERS[name](fn)
    if failure is None:
        print("✅ %s works on %d tests. %s" % (name, len(_P3_TESTS[name]), _P3_SUCCESS[name]))
        return
    print("❌ " + failure[0])
    print("   💡 Hint: " + failure[1])


def check_slope_by_hop(fn):
    """Task A: tests slope_by_hop on the game's hop, two other hops, and other numbers, guesses and ratings."""
    try:
        _p3_check("slope_by_hop", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 3 again, then this cell." % error)


def check_slope(fn):
    """Task B: tests slope on the four numbers of two guesses, on another rating, on decimals and at a loss of 0."""
    try:
        _p3_check("slope", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 3 again, then this cell." % error)


def check_gradient_step(fn):
    """Task C: tests gradient_step on six guesses, with three step sizes: the guesses after it and the loss."""
    try:
        _p3_check("gradient_step", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 3 again, then this cell." % error)


def check_gradient_descent(fn):
    """Task D: tests gradient_descent on seven runs: the list of losses, the stop, max_steps, the end guesses."""
    try:
        _p3_check("gradient_descent", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 3 again, then this cell." % error)


if _p3_missing:
    print("Note: part 1 has not run in this session, so this cell defined %s for part 3. Part 3 works on its own; "
          "to run part 1 as well, click on this cell, choose Runtime > Run before in the menu, then run this cell "
          "again." % " and ".join(_p3_missing))
print("✅ Setup of part 3 complete.")

## The loss, given again

Part 3 measures the loss of a guess again and again. Here are part 2's `squared_loss` and `corner_loss`, as its
solutions write them, given again so that this part works even if your part 2 is not finished. Run the cell.

In [ ]:
# Given again: part 2's squared_loss and corner_loss, as its solutions write them, so that part 3 works
# on its own, even when part 2 is not finished
def squared_loss(rating, prediction):
    # rating and prediction are two numbers. Return the difference times itself.
    difference = rating - prediction
    loss = difference * difference
    return loss


def corner_loss(guesses, rating):
    # guesses[0] is Ida's row of two numbers, guesses[1] is Paddington's row.
    prediction = inner_product(guesses[0], guesses[1])
    loss = squared_loss(rating, prediction)
    return loss

## The stuck guess

Part 2's loop stopped on this guess: Ida -3 and -3, *Paddington* -3 and 2. The prediction is 3 and the rating 2, so the
loss is 1. A hop of 1 on a number adds its partner to the prediction or takes it away, and the partners here
are 2 and 3 in size: the prediction jumps from 3 to 0, 1, 5 or 6, over the rating or away from it. No whole hop lowers the
loss, so the loop could not move.

In [ ]:
# Part 2's corner, Ida and Paddington, and the guess where part 2's loop got stuck
rating = 2                                   # Ida gave Paddington 2: ratings[0][0] in part 2
guesses = [[-3, -3], [-3, 2]]                # Ida's two numbers on top, Paddington's two below
corner_names = ["Ida", "Paddington"]
number_names = ["first number", "second number"]
show_matrix(guesses, corner_names, number_names)
print("The prediction:", inner_product(guesses[0], guesses[1]))
print("The loss:      ", corner_loss(guesses, rating))

## A valley between the hops

In the game the grasshopper stands on Ida's first number and watches only that number, with the other three held: the loss
against that one number is its **ground**. The cell below walks the ground at the two whole hops of part 2 and at
the stuck guess between them.

In [ ]:
# The ground of Ida's first number: the loss when only that number changes, the other three held
for x in [-4, -3, -2]:
    trial = [[x, -3], [-3, 2]]
    print("Ida's first number", x, "  the loss", corner_loss(trial, rating))

**Reading the result.** The two whole hops land at the losses 16 and 4, both higher than 1. Between -3 and -2
the ground dips into a valley, and its bottom, the loss 0, lies at about -2.67. Hops of 1 jump over it. Smaller
hops can land in it.

## The grasshopper's hop

💭 **Think first.** The grasshopper stands on Ida's first number, -3, and hops a tiny step to the right, eps = 0.01. Will the
loss go up or down? Then run the cell below. It makes the hop by hand on a copy of the stuck guess, `trial`, and
measures the loss before and after it.

In [ ]:
# The grasshopper's first hop, by hand: Ida's first number, from -3 to -3 + 0.01
trial = [[-3, -3], [-3, 2]]                  # the stuck guess again
loss_before = corner_loss(trial, rating)
trial[0][0] = -3 + 0.01                      # the hop
loss_after = corner_loss(trial, rating)
print("The loss before the hop:   ", loss_before)
print("The loss after the hop:    ", round(loss_after, 4))
print("The change:                ", round(loss_after - loss_before, 4))
print("The change per unit of hop:", round((loss_after - loss_before) / 0.01, 4))

**Reading the result.** The loss fell from 1 to 0.9409, so the ground goes down to the right. The change is 0.9409 - 1
= -0.0591 for a hop of 0.01, and per unit of hop that is -0.0591 / 0.01 = -5.91: the **slope** of the ground where the grasshopper
stands. A negative slope means downhill to the right.

`round(x, 4)` rounds a number to 4 decimals for printing. Python computes with binary fractions, so without it
the loss after the hop prints as 0.9409000000000013.

## 🎯 Task A. The slope from a tiny hop

Write `slope_by_hop(guesses, rating, row, position, eps)`, the grasshopper's measurement on the number
`guesses[row][position]`, the way the cell above did it by hand: the loss before, the hop by `eps`, the loss
after, the hop back, and the change divided by `eps`. The grasshopper only feels the ground, so the function hops
back and leaves `guesses` as it found them. `row` and `position` pick the number as in part 2: `guesses[0][0]` is
Ida's first number. For it, `slope_by_hop(guesses, rating, 0, 0, 0.01)` gives -5.91, with a little noise in the last decimals.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# Four moves, as in the cell that hopped by hand:
# 1. the hop: the number becomes original + eps
# 2. the loss after the hop: corner_loss(guesses, rating)
# 3. the hop back: the number becomes original again
# 4. the change, the loss after minus the loss before, divided by eps

In [ ]:
#@title 💡 Hint 2 for task A: the shape of the code (double-click to open) { display-mode: "form" }
#     guesses[row][position] = original + eps
#     loss_after = corner_loss(guesses, rating)
#     guesses[row][position] = ...                 # the hop back, to the original
#     change = loss_after - loss_before
#     return ...                                   # the change per unit of hop

In [ ]:
# 🎯 YOUR TURN, task A: the slope from a tiny hop
def slope_by_hop(guesses, rating, row, position, eps):
    # The grasshopper stands on the number guesses[row][position]. It hops by eps, measures how much
    # the loss changed, and hops back. Return the change per unit of hop: the change divided by eps.
    loss_before = corner_loss(guesses, rating)     # the loss where the grasshopper stands
    original = guesses[row][position]              # the number as it is now
    ...                                            # 🎯 YOUR TURN: the hop, make the number original + eps
    loss_after = ...                               # 🎯 YOUR TURN: the loss after the hop (use corner_loss)
    ...                                            # 🎯 YOUR TURN: the hop back, put the original back
    change = ...                                   # 🎯 YOUR TURN: the loss after minus the loss before
    return ...                                     # 🎯 YOUR TURN: the change divided by eps

In [ ]:
check_slope_by_hop(slope_by_hop)

In [ ]:
# The grasshopper's hop of the game, with your function, on Ida's first number
eps = 0.01
print_hop(eps, slope_by_hop(guesses, rating, 0, 0, eps))

**Reading the result.** The function measures -5.91, the slope the cell above found by hand. It is negative, so
the ground goes down to the right, and its size, 5.91, says how steep the ground is there.

## 🎯 Task B. The slope from a formula

In the game, the calculation with ε showed what the grasshopper measures (ε is `eps` in the code). A hop of ε
on Ida's first number changes only one product in the stack, by ε times its partner, Paddington's first number, -3. The prediction becomes 3 - 3ε, the
difference (the rating minus the prediction) becomes -1 + 3ε, and the loss 1 - 6ε + 9ε². So the change of the loss per unit of
hop is -6 + 9ε. With the game's hop, ε = 0.01, that is -5.91, the slope above. The part 9ε comes from the size of the hop.
Without it the slope is -6, and -6 is -2 times the partner (-3) times the difference (-1).

The same holds for every guessed number: **the slope is -2 times its partner times the difference.** The partner
of a number is the number it meets in the stack, at the same position in the other row:
`guesses[1 - row][position]`. For row 0 the other row is 1, and for row 1 it is 0.

Write `slope(guesses, rating, row, position)`. For Ida's first number at the stuck guess it gives -6.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# The difference is the rating minus the prediction: 2 - 3 = -1 at the stuck guess.
# The partner of guesses[row][position] sits in the other row, 1 - row, at the same position.
# For Ida's first number, guesses[0][0], the partner is guesses[1][0], Paddington's first number.

In [ ]:
#@title 💡 Hint 2 for task B: the shape of the code (double-click to open) { display-mode: "form" }
#     difference = rating - prediction
#     partner = guesses[1 - row][position]
#     return -2 * ... * ...                       # -2 times the partner times the difference

In [ ]:
# 🎯 YOUR TURN, task B: the slope from a formula
def slope(guesses, rating, row, position):
    # The slope of the loss at the number guesses[row][position]: -2 times its partner times the
    # difference. Its partner is the number it meets in the stack: guesses[1 - row][position].
    prediction = inner_product(guesses[0], guesses[1])
    difference = ...                               # 🎯 YOUR TURN: the rating minus the prediction
    partner = ...                                  # 🎯 YOUR TURN: the partner of guesses[row][position]
    return ...                                     # 🎯 YOUR TURN: -2 times the partner times the difference

In [ ]:
check_slope(slope)

In [ ]:
# The four slopes at the stuck guess: from the formula, and from the game's hop of 0.01
for row in range(2):
    for position in range(2):
        from_formula = slope(guesses, rating, row, position)
        from_hop = slope_by_hop(guesses, rating, row, position, 0.01)
        print_slopes(row, position, from_formula, from_hop)

**Reading the result.** The formula gives -6, 4, -6 and -6, and the hop of 0.01 agrees to within 0.1. Three slopes are negative:
those numbers should grow. Ida's second number has the slope 4: its partner, Paddington's second number, is 2, so a hop up makes the loss bigger, and
this number should shrink. A slope says which way is downhill, and how steep it is.

## 🎯 Task C. One step downhill

The grasshopper jumps against the slope: **the new number is the old number minus alpha times its slope.** With
a negative slope the ground goes down to the right, so the number grows; with a positive slope it shrinks. alpha
is the size of the step, 0.02 in the game: small, because a slope can be large, and a step too far lands on the
other side of the valley. For Ida's first number: `-3 - 0.02 * (-6)` is -2.88.

All four numbers move in one step, and every slope is measured on the same guess, before any number moves. So
the function has two loops: the first measures all four slopes, the second moves all four numbers. The four
slopes together, in the places of the four numbers, are called the **gradient**: [[-6, 4], [-6, -6]] at the stuck guess.

Write `gradient_step(guesses, rating, alpha)`. It changes `guesses` itself and returns the loss after the step.
From the stuck guess with alpha 0.02: Ida -2.88 and -3.08, Paddington -2.88 and 2.12, and the loss 0.0553.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# The first loop only measures: slopes[row][position] = slope(guesses, rating, row, position).
# No number moves in it, so all four slopes belong to the same guess.
# The second loop moves: each number takes alpha times its slope away from itself.

In [ ]:
#@title 💡 Hint 2 for task C (double-click to open) { display-mode: "form" }
# For Ida's first number at the stuck guess: -3 - 0.02 * (-6) = -2.88.
# In Python, for any number: guesses[row][position] - alpha * slopes[row][position]

In [ ]:
#@title 💡 Hint 3 for task C: the shape of the code (double-click to open) { display-mode: "form" }
#             slopes[row][position] = slope(guesses, rating, row, position)
#     ...
#             guesses[row][position] = guesses[row][position] - ... * ...

In [ ]:
# 🎯 YOUR TURN, task C: one step downhill for all four numbers
def gradient_step(guesses, rating, alpha):
    # One step downhill for all four numbers. First measure the four slopes on the guess as it is
    # now; then move every number against its slope: the old number minus alpha times its slope.
    # Return the loss after the step.
    slopes = [[0, 0], [0, 0]]                      # room for the four slopes, in the places of the numbers
    # First: all four slopes, before any number moves
    for row in range(2):
        for position in range(2):
            slopes[row][position] = ...            # 🎯 YOUR TURN: the slope of guesses[row][position] (use slope)
    # Then: every number moves against its slope
    for row in range(2):
        for position in range(2):
            ...                                    # 🎯 YOUR TURN: the number minus alpha times its slope
    return corner_loss(guesses, rating)            # the loss after the step

In [ ]:
check_gradient_step(gradient_step)

In [ ]:
# One step downhill from the stuck guess, with the step size of the game
alpha = 0.02
guesses = [[-3, -3], [-3, 2]]                # the stuck guess again
loss_after = gradient_step(guesses, rating, alpha)
print_step(1, guesses, loss_after)
show_matrix(guesses, corner_names, number_names)

**Reading the result.** Every number moved a little against its slope: Ida -2.88 and -3.08, Paddington -2.88 and 2.12. The loss fell from 1 to 0.0553 in one
step, from a guess where whole hops could not move at all. The prediction is now 1.7648, a little below the rating
2.

## 🎯 Task D. Again and again

Gradient descent repeats the step: measure the slopes, step downhill, and go again. The loss shrinks step after
step, and below 0.0001 it counts as 0. Write `gradient_descent(guesses, rating, alpha, max_steps)`. It makes at most
`max_steps` steps, with `for step_number in range(max_steps)`, and stops early, with `return`, once the loss is
below 0.0001. It changes `guesses` itself and returns the list of losses, the loss at the start first, so that you
can watch it fall. A given line, `print_step`, prints each step: the loss, the prediction and the four numbers.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task D (double-click to open) { display-mode: "form" }
# losses starts as a list that holds one number, the loss at the start: losses = [loss].
# Each round of the loop does three things: first the stop, then one step, then the new loss
# goes at the end of the list.

In [ ]:
#@title 💡 Hint 2 for task D (double-click to open) { display-mode: "form" }
# The stop: if loss < 0.0001: return losses. It comes first in the round, so a guess that is
# already good makes no step at all.
# One step: loss = gradient_step(guesses, rating, alpha) moves the four numbers and gives back
# the loss after the step.

In [ ]:
#@title 💡 Hint 3 for task D: the shape of the code (double-click to open) { display-mode: "form" }
#     losses = [loss]
#     for step_number in range(max_steps):
#         if loss < 0.0001:
#             return losses
#         loss = gradient_step(guesses, rating, alpha)
#         losses.append(...)
#         print_step(step_number + 1, guesses, loss)
#     return losses

In [ ]:
# 🎯 YOUR TURN, task D: gradient descent on the corner
def gradient_descent(guesses, rating, alpha, max_steps):
    # Step downhill again and again. Stop when the loss is below 0.0001, where it counts as 0, or
    # after max_steps steps. Return the list of losses, the loss at the start first.
    loss = corner_loss(guesses, rating)            # the loss at the start
    losses = ...                                   # 🎯 YOUR TURN: a list that holds the loss at the start
    for step_number in range(max_steps):
        # 🎯 YOUR TURN: if the loss is below 0.0001, return losses
        ...
        loss = ...                                 # 🎯 YOUR TURN: one step downhill (use gradient_step)
        ...                                        # 🎯 YOUR TURN: put the new loss at the end of losses
        print_step(step_number + 1, guesses, loss)
    return losses

In [ ]:
check_gradient_descent(gradient_descent)

In [ ]:
# Gradient descent from the stuck guess, at most 100 steps
guesses = [[-3, -3], [-3, 2]]                # the stuck guess again
losses = gradient_descent(guesses, rating, alpha, 100)
show_losses(losses)
show_matrix(guesses, corner_names, number_names)

**Reading the result.** 4 steps, and the loss goes 1, 0.0553, 0.0028, 0.00014, then about 0. The prediction swings around the rating, 1.76, 2.05, 1.99 and 2.00, and settles on
2. Part 2's what-if loop could not move from this guess: its whole hops jumped over the valley. The small steps,
guided by the slopes, land in it.

## Five experiments

💭 **Think first** before you run each cell below: what will the loss do?

1. Part 2's first guess, Ida -1 and -2 with *Paddington* -1 and 2, the loss 25. Part 2's loop needed 3 whole hops from
   there. How many steps will gradient descent need?
2. Your own guess from the game.
3. All four numbers at 0, the loss 4. What slopes will the grasshoppers feel?
4. A step that is too big: alpha 0.1 in place of 0.02, from the stuck guess.
5. A flipped guess: Paddington's numbers are Ida's with the signs flipped, Ida 1 and 2 with *Paddington* -1 and -2.

In [ ]:
# Experiment 1: part 2's first guess
first = [[-1, -2], [-1, 2]]
losses = gradient_descent(first, rating, alpha, 100)
show_losses(losses)

**Reading the result.** 16 steps, from 25 to about 0. Part 2's loop needed only 3 whole hops here, because a guess of
whole numbers, Ida -2 and -1 with *Paddington* -2 and 2, gives the rating exactly. Gradient descent takes more steps,
all of them small, and it also gets out of the stuck guess, where whole hops could not move.

**Experiment 2, your own guess.** In section 2 of the game you typed a guess of your own for Ida and *Paddington*. Type
its four numbers into the cell below, in place of the Matchmaker's first guess, and run the cell. If you skipped
that step of the game, pick any four numbers from -2 to 2. Run the cell again whenever you change the numbers.

### 🎯 Your turn

In [ ]:
# 🎯 YOUR TURN: your own guess from the game, in place of the Matchmaker's first guess
my_guesses = [[-1, -2],      # 🎯 YOUR TURN: Ida's first and second number
              [-1, 2]]       # 🎯 YOUR TURN: Paddington's first and second number
losses = gradient_descent(my_guesses, rating, alpha, 50)
show_losses(losses)
show_matrix(my_guesses, corner_names, number_names)

**Reading the result.** The loss never rises from one step to the next. From every guess of numbers from -2 to
2 it gets below 0.0001 within 42 steps, except from 25 guesses: the flipped guesses of experiment 5, all zeros among
them. Where your numbers end depends on where they start: many pairs of rows give the inner product 2.

In [ ]:
# Experiment 3: all four numbers at 0, 5 steps
zeros = [[0, 0], [0, 0]]
losses = gradient_descent(zeros, rating, alpha, 5)
show_losses(losses)

**Reading the result.** The loss stays 4. At 0 every partner is 0, so every slope, -2 times the partner times
the difference, is 0: the ground is flat there, and the grasshoppers feel no slope to follow. No number moves.
Part 2's loop was stuck here as well.

In [ ]:
# Experiment 4: a step that is too big, alpha = 0.1, 3 steps from the stuck guess
trial = [[-3, -3], [-3, 2]]
losses = gradient_descent(trial, rating, 0.1, 3)
show_losses(losses)

**Reading the result.** The losses grow: 1, 25.8064, 486.5594, 77270.0032. With alpha 0.1 the first step takes the prediction from 3 to -3.08, far
past the rating 2, and the loss lands at 25.8. There the slopes are steeper, so the next jump is longer still. A
small alpha, 0.02, makes small jumps, and the loss falls at every step.

In [ ]:
# Experiment 5: Paddington's numbers are Ida's with the signs flipped, 30 steps
flipped = [[1, 2], [-1, -2]]
losses = gradient_descent(flipped, rating, alpha, 30)
show_losses(losses)
show_matrix(flipped, corner_names, number_names)

**Reading the result.** The loss falls at every step, from 49 down towards 4, and stays above 0.0001: the numbers
of both rows slide towards 0. 💭 Why does it stop near 4? Think first, then open the folded answer below.

In [ ]:
#@title 💭 Answer: why the flipped guess stops near the loss 4 (double-click to open) { display-mode: "form" }
# Paddington's row is Ida's row with every sign flipped, so both products in the stack are negative,
# and the prediction is below 0, far from the rating 2.
# Every step shrinks both rows by the same factor, and they stay flipped, so the prediction
# can never climb to 2. The best a flipped pair can do is all zeros, where the prediction is 0
# and the loss is 4. There every slope is 0: flat ground, as in experiment 3.
# Part 2's whole hops get out of such a guess: a hop of 1 on one number breaks the flip. From
# every flipped guess of numbers from -2 to 2 except all zeros, part 2's loop reached the loss 0.

## Back to the game

That is part 3: your `gradient_descent` feels the slope of every number and steps downhill, a small step at a
time, and gets out of the stuck guess. Section 4 of the game, "Many customers, many films", goes from the corner
to the whole table with the same method, one grasshopper for every guessed number. Part 4 of this notebook
comes after it.